<a href="https://colab.research.google.com/github/alexeysher/skillbox-computer-vision-project/blob/main/emotion_recognition_1_2_1024_2048_0_0_0_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Skillbox. Дипломная работа по компьютерному зрению. Распознавание эмоций человека.

## Настройки

### Основные

In [ ]:
PROJECT_NAME = 'skillbox-computer-vision-project' # Название проекта
LOCAL_PROJ_PATH = f'D:/{PROJECT_NAME}' # Путь к папке проекта на локальном компьютере
COLAB_PROJ_PATH = f'/content/{PROJECT_NAME}' # Путь к папке проекта в сессионном хранилище Google Colab
LOCAL_GD_PROJ_PATH = f'G:/Мой диск/{PROJECT_NAME}' # Путь к папке проекта на Google Диске'е на локальном компьютере
COLAB_GD_PROJ_PATH = f'/content/drive/MyDrive/{PROJECT_NAME}' # Путь к папке проекта на Google Диск'е в Google Colab
TRAIN_DATASET_PATH = 'train' # Путь к исходному тренировочному датасету внутри папки проекта
TEST_DATASET_PATH = 'test_kaggle' # Путь к исходному тестовому датасету внутри папки проекта
TRAIN_DATASET_URL = 'https://drive.google.com/file/d/1TG9P5B2k3eTbC4XDxDmEc07dyAORPC16/view?usp=sharing' # Ссылка на архив тренировочного датасета
TRAIN_DATASET_EXT = 'zip' # Тип (расширение файла) архива тренировочного датасета
TEST_DATASET_URL = 'https://drive.google.com/file/d/12QrDrLT1F-X7UycvOoApXFqxTw3Zx93K/view?usp=sharing' # Ссылка на архив тестового датасета
TEST_DATASET_EXT = 'zip' # Тип (расширение файла) архива тестового датасета
KAGGLE_API_TOKEN_URL = 'https://drive.google.com/file/d/1EzdFbMawx7bmAU9ze_sxL-1yBf2SPrg-/view?usp=sharing' # Ссылка на токен для подключения к платформе Kaggle через API
MAX_INFERENCE_TIME = .33 # Максимально допустимое время инференса модели в секундах
INFERENCE_TIME_WEIGHT = .6 # Вес времени инференса при выборе базовой модели
BASE_MODEL_MAX_SIZE = 64 # Максимально допустимый размер базовой модели в МБ
BASE_MODEL_POOLINGS = 'avg' # Тип пулинга на выходе базовой моделей ('avg' - average, 'max' - max)
MODEL_ON_TOP_DENSE_NUMS = [1, 2] # Варианты количества дополнительных полносвязных слоев
MODEL_ON_TOP_DENSE_UNITS = [1024, 2048] # Варианты количества выходных нейронов в дополнительном полносвязном слое
MODEL_ON_TOP_DROPOUT_RATES = [.0, .2] # Варианты доли исключаемых данных перед подачей в полносвязный слой при обучении
OPTIMIZER = 'Adam' # Название оптимизатора, используемого при обучении модели
MODEL_ON_TOP_INITIAL_LEARNING_RATE = 1e-4 # Начальная скорость обучения полносвязной модели
MODEL_ON_TOP_LEARNING_RATE_DECAY_RATE = 0.96 # Коэффициент изменения скорости обучения полносвязной модели по окончании каждой эпохи
MODEL_INITIAL_LEARNING_RATE = 2e-5 # Начальная скорость обучения модели при тонкой настройке
MODEL_LEARNING_RATE_DECAY_RATE = 0.96 # Коэффициент изменения скорости обучения модели по окончании каждой эпохи при тонкой настройке
RANDOM_FLIP = 'horizontal' # Тип случайного зеркального отражения изображения
RANDOM_ZOOM = 0. # Максимальное изменение масштаба изображения
RANDOM_ROTATION_FACTOR = .1 # Максимальный поворт изображения (в долях от полного оборота - 360°)
RANDOM_CONTRACT_FACTOR = .2 # Максимальное изменение контраста (в долях от исходного значения)
RANDOM_BRIGHTNESS_FACTOR = .2 # Максимальное изменение яркости (в долях от исходного значения)
SEED = 123 # Инициализатор генератора случайных чисел
VERBOSE = 1 # Режим верболизации (0-тийхий, 1-вывод сообщений)

### Описание эмоций

Если используется разметка данных с помощью порядковых номеров эмоций, то названия эмоций должны быть перечислены в виде списка или кортежа. 

При использовании разметки в виде Valence-Arousal описание эмоций должно быть представлено в виде словаря. В качестве ключей словаря должны использоваться названия эмоций. Значениями словаря должны быть пары чисел, характеризирующие уровень Valence и Arousal эмоций. Уровни Valence и Arousal должны быть числами в диапазоне от -1.0 до 1.0 включительно. 

В обоих случаях перечисление эмоций должен осуществляться в алфавитном порядке.

In [ ]:
EMOTIONS = (
    'anger', # гнев, злость
    'contempt', # презрение
    'disgust', # отвращение
    'fear', # страх
    'happy', # веселый
    'neutral', # нейтральный
    'sad', # грусть
    'surprise', # удивленность
    'uncertain', # неуверенность
)

### Список базовых моделей из Keras Applications с указанием [справочных данных](https://keras.io/api/applications/):
- размер в МБ (Size (MB))
- точность предсказаний в % (Top-1 Accuracy)

In [ ]:
KERAS_BASE_MODELS = {
    'MobileNet': (16, 70.40),
    'MobileNetV2': (14, 71.30),
    'NASNetMobile': (23, 74.40),
    'InceptionV3': (92, 77.90),
    'ResNet50V2': (98, 76.00),
    'EfficientNetB0': (29, 77.10),
    'ResNet50': (98, 74.90),
    'EfficientNetB1': (31, 79.10),
    'VGG16': (528, 71.30),
    'ResNet101V2': (171, 77.20),
    'DenseNet121': (33, 75.00),
    'EfficientNetB2': (36, 80.10),
    'VGG19': (549, 71.30),
    'ResNet101': (171, 76.40),
    'DenseNet169': (57, 76.20),
    'ResNet152V2': (232, 78.00),
    'Xception': (88, 79.00),
    'DenseNet201': (80, 77.30),
    'ResNet152': (232, 76.60),
    'InceptionResNetV2': (215, 80.30),
    'EfficientNetB3': (48, 81.60),
    'EfficientNetB4': (75, 82.90),
    'NASNetLarge': (343, 82.50),
    'EfficientNetB5': (118, 83.60),
    'EfficientNetB6': (166, 84.00),
    'EfficientNetB7': (256, 84.30),
    'EfficientNetV2B0': (29, 78.70),
    'EfficientNetV2B1': (34, 79.80),
    'EfficientNetV2B2': (42, 80.50),
    'EfficientNetV2B3': (59, 82.00),
    'EfficientNetV2S': (88, 83.90),
    'EfficientNetV2M': (220, 85.30),
    'EfficientNetV2L': (479, 85.70),
}

### Пайплан предобработки изображений

In [ ]:
IMAGE_PREPROCESSING_PIPELINE = {
    'name': 'image_preprocessing',
    'description': 'Пайплан предобработки изображений',
    'report_csv': 'pipeline_images_preprocessing.csv',
    'stages': 
    [
        {
            'name': 'train_face_extraction',
            'description': 'Извлечение изображений лиц из тренировочного датасета',
            'platform': 'local',
            'params': {
                'path': 'train_faces', # Путь к папке тренировочного датасета с изображениями лиц
                'engines': 8, # Количество параллельно работающих "движков"
                'batch_size': 125, # Размер батча
                'scale_factor': 0.709, # Фактор масштабирования при детектировании лиц на изображении
                'min_face_size': 128, # Минимальный размер лица при детектировании лиц на изображении
                'process_csv': 'train_face_extraction_process.csv', # Путь к файлу с детальной информацией
                'result_csv': 'train_face_extraction.csv', # Путь к файлу с результатами
            },
        },
        {
            'name': 'test_face_extraction',
            'description': 'Извлечение изображений лиц из тестового датасета',
            'platform': 'local',
            'params': {
                'path': 'test_faces', # Путь к папке тестового датасета с изображениями лиц
                'engines': 8, # Количество параллельно работающих "движков"
                'batch_size': 125, # Размер батча
                'scale_factor': 0.709, # Фактор масштабирования при детектировании лиц на изображении
                'min_face_size': 128, # Минимальный размер лица при детектировании лиц на изображении
                'process_csv': 'test_face_extraction_process.csv', # Путь к файлу с детальной информацией
                'result_csv': 'test_face_extraction.csv', # Путь к файлу с результатами
            }
        },
        {
            'name': 'train_balancing',
            'description': 'Балансировка тренировочного датасета',
            'platform': 'local',
            'params': {
                'path': 'train_balanced_faces', # Путь к папке сбалансированного датасета
                'engines': 8, # Количество параллельно работающих "движков"
                'batch_size': 125, # Размер батча
                'process_csv': 'train_balancing_process.csv', # Путь к файлу с детальной информацией
                'result_csv': 'train_balancing.csv', # Имя файла с результатами агументациии тренировочного датасета
            },
        },
    ]
}

### Пайплан сбора информации о базовых моделях в Keras Applications

In [ ]:
KERAS_BASE_MODELS_PROCESSING_PIPELINE = {
    'name': 'keras_base_models_processing',
    'description': 'Пайплайн сбора информации о базовых моделях в Keras Applications',
    'report_csv': 'pipeline_base_models_processing.csv',
    'stages': [
        {
            'name': 'sizes_retrieving',
            'description': 'Получение информации о размерах входных изображений и векторов признаков',
            'platform': 'colab', # Выполняется в Google Colab
            'params': {
               'result_csv': 'base_model_sizes.csv', # Путь к файлу с отобранными моделями
            }
        },
        {
            'name': 'inference_time_measuring',
            'description': 'Измерение времени инференса моделей',
            'platform': 'colab', # Выполняется в Google Colab
            'params': {
                'batch_size': 1, # Размер батча
                'batches': 1, # Количество батчей в датасете
                'repetitions': 100, # Количество повторов
                'result_csv': 'model_inference_times.csv', # Путь к файлу с отобранными моделями
            }
        },
        {
            'name': 'base_model_selection',
            'description': 'Выбор базовой модели',
            'platform': 'colab', # Выполняется в Google Colab
            'params': {
                'inference_time_weight': INFERENCE_TIME_WEIGHT, # Вес времени инференса при выборе базовой модели
                'top1_accuracy_weight': 1 - INFERENCE_TIME_WEIGHT, # Вес точности  при выборе базовой модели
                'process_csv': 'base_model_selection.csv', # Путь к файлу с данными процесса выбора базовой модели
                'result_csv': 'base_model.csv', # Путь к файлу с описанием выбранной базовой модели
            }
        },
    ]
}

### Пайплан создания модели

In [ ]:
MODEL_BUILDING_PIPELINE = {
    'name': 'model_building',
    'description': 'Пайплайн создания модели',
    'report_csv': 'pipeline_model_building.csv',
    'stages': [
        {
            'name': 'train_feature_extraction',
            'description': 'Извлечение признаков из тренировочного датасета',
            'platform': 'colab', # Выполняется в Google Colab
            'params': {
                'path': 'train_features', # Путь к папке с файлами батчей извлеченных признаков
                'flip': 'horizontal', # Случайное отрезкаливание изображения
                'rotation_factor': 0.1, # Фактор случайного поворота (против или по часой стрелке) изображения при аугментации, доли от 360°
                'zoom_factor': 0.2, # Фактор случайного приближения или удаления изображения при аугментации
                'contrast_factor': 0.2, # Фактор случаного изменения контраста изображения
                'brightness_factor': 0.2, # Фактор случаного изменения яркости изображения
                'batch_size': 256, # Размер батча
                'buffer_size': 10, # Размер буфера
            }
        },
        {
            'name': 'test_feature_extraction',
            'description': 'Извлечение признаков из тестового датасета',
            'platform': 'colab', # Выполняется в Google Colab
            'params': {
                'path': 'test_features', # Путь к папке с файлами батчей извлеченных признаков
                'flip': 'horizontal', # Случайное отрезкаливание изображения
                'rotation_factor': 0.1, # Фактор случайного поворота (против или по часой стрелке) изображения при аугментации, доли от 360°
                'zoom_factor': 0.2, # Фактор случайного приближения или удаления изображения при аугментации
                'contrast_factor': 0.2, # Фактор случаного изменения контраста изображения
                'brightness_factor': 0.2, # Фактор случаного изменения яркости изображения
                'batch_size': 256, # Размер батча
                'buffer_size': 10, # Размер буфера
            }
        },
        {
            'name': 'model_on_top_selection',
            'description': 'Выбор лучшей полносвязной модели',
            'platform': 'colab',
            'params': {
                'path': 'model_on_top_selection', # Путь к папке с логами и весами полносвязной модели
                'batch_size': 64, # Размер батча
                'optimizer_name': OPTIMIZER, # Оптимизатор,
                'initial_learning_rate': MODEL_ON_TOP_INITIAL_LEARNING_RATE, # Начальная скорость обучения
                'learning_rate_decay_rate': MODEL_ON_TOP_LEARNING_RATE_DECAY_RATE, # Коэффициент снижения скорости обучения
                'epochs': 100, # Количество эпох при измерении времени инференса
                'patience': 10, # Макс. количество эпох без улучшения точности
                'process_csv': 'model_on_top_selection.csv', # Путь с результатами обучения моделей
                'result_csv': 'selected_model_on_top.csv', # Путь к файлу с описанием выбранной базовой модели
            }
        },
        {
            'name': 'model_on_top_training',
            'description': 'Обучение полносвязных моделей',
            'platform': 'colab',
            'params': {
                'path': 'model_on_top_training', # Путь к папке с логами и весами полносвязной модели
                'flip': 'horizontal', # Случайное отрезкаливание изображения
                'rotation_factor': 0.1, # Фактор случайного поворота (против или по часой стрелке) изображения при аугментации, доли от 360°
                'zoom_factor': 0.2, # Фактор случайного приближения или удаления изображения при аугментации
                'contrast_factor': 0.2, # Фактор случаного изменения контраста изображения
                'brightness_factor': 0.2, # Фактор случаного изменения яркости изображения
                'batch_size': 32, # Размер батча
                'buffer_size': 100, # Размер буфера
                'optimizer_name': OPTIMIZER, # Оптимизатор,
                'initial_learning_rate': MODEL_ON_TOP_INITIAL_LEARNING_RATE, # Начальная скорость обучения
                'learning_rate_decay_rate': MODEL_ON_TOP_LEARNING_RATE_DECAY_RATE, # Коэффициент снижения скорости обучения
                'epochs': 2, # Количество эпох при измерении времени инференса
                'epochs_per_run': 2, # Количество эпох обучения за один запуск
                'patience': 1, # Макс. количество эпох без улучшения точности
                'process_csv': 'model_on_top_training.csv', # Путь с результатами обучения моделей
                'result_csv': 'trained_model_on_top.csv', # Путь к файлу с описанием выбранной базовой модели
            }
        },
        {
            'name': 'model_fine_tuning',
            'description': 'Тонкая настройка модели',
            'platform': 'colab', # Выполняется в Google Colab
            'params': {
                'path': 'model_fine_tuning', # Путь к папке с логами и весами полносвязной модели
                'batch_size': 32, # Размер батча
                'buffer_size': 100, # Размер буфера
                'optimizer_name': OPTIMIZER, # Оптимизатор
                'initial_learning_rate': MODEL_INITIAL_LEARNING_RATE, # Начальная скорость обучения
                'learning_rate_decay_rate': MODEL_LEARNING_RATE_DECAY_RATE, # Коэффициент снижения скорости обучения
                'epochs': 50, # Количество эпох обучения
                'epochs_per_run': 10, # Количество эпох обучения за один запуск
                'patience': 10, # Макс. количество эпох без улучшения точности
                'process_csv': 'model_fine_tuning.csv', # Путь к файлу с данными процесса выбора базовой модели
                'result_csv': 'model.csv', # Путь с
            }
        },
        {
            'name': 'model_deploy_test',
            'description': 'Тестирование работы модели',
            'platform': 'local', # Выполняется в Google Colab
            'params': {
                'path': 'model_deploy_test',
                'scale_factor': 0.209, # Фактор масштабирования при детектировании лиц на изображении
                'min_face_size': 128, # Минимальный размер лица при детектировании лиц на изображении
                'min_probability': 0.6, # Минимальная вероятность предсказания эмоции
                'max_error': 0.25, # Максимальная ошибка предсказания valence-arousal
                'process_csv': 'model_deploy_test.csv', # Путь к файлу с данными процесса выбора базовой модели
                'result_csv': 'emotion_files.csv', # Путь к файлу с описанием выбранной базовой модели
            }
        },
    ]
}

## Подготовка

### Определение платформы запуска ноутбука (локальный компьютер или Google Colab)

In [ ]:
if 'google.colab' in str(get_ipython()):
    platform = 'colab'  
    print('Notebook is running on Google Colab.')
else: 
    platform = 'local'
    print(f'Notebook is running locally.')

Notebook is running on Google Colab.


### Установка и загрузка необходимых библиотек

#### Установка необходимых библиотек при их отсутствии

In [ ]:
from importlib.util import find_spec

# Функция устанавливает пакет при его отсутствии
def install_package(name: str):
    if not find_spec(name):
        !pip install {name}

# Список пакетов
packages = [
    'validators',
    'mtcnn', # Детектор лиц, разработанный Iván de Paz Centeno ipazc@unileon.es.
    'ipyparallel', # Пакет Python и набор CLI скриптов для управления кластерами процессов IPython, построенная на протоколе Jupyter
    'tqdm',
    'numpy',
    'pandas',
    'gdown',
    'matplotlib',
    'tensorflow',
    'kaggle',
    'cv2',
]
if platform == 'local':
    packages += ['jupyterlab_widgets', 'ipywidgets', 'widgetsnbextension']

# Установка пакетов
for package in packages:
    install_package(package)

#### Импорт необходимых библиотек

In [ ]:
from typing import Optional, Union, Tuple, List, Dict
from IPython.display import display, Image as DisplayImage
import inspect
import itertools
import validators
from time import sleep
from datetime import datetime
from timeit import timeit
from pathlib import Path
import shutil
import gdown
import ipyparallel as ipp
from tqdm.notebook import tqdm, trange
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
import cv2 as cv
from copy import deepcopy
from PIL import Image, ImageOps
import mtcnn
with open(mtcnn.mtcnn.__file__, 'r') as file:
    s = file.read()
if s.find('verbose') == -1:
    s = s.replace('predict(img_y)', 'predict(img_y, verbose=0)')
    s = s.replace('predict(tempimg1)', 'predict(tempimg1, verbose=0)')
    with open(mtcnn.mtcnn.__file__, 'w') as file:
        file.write(s)
from mtcnn.mtcnn import MTCNN
import tensorflow as tf
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
from tensorflow.keras import models, layers, activations, optimizers, metrics, losses, callbacks, utils, applications, initializers
if platform == 'colab':
    from psutil import virtual_memory
    from google.colab import output
    output.enable_custom_widget_manager()

#### Импорт необходимых расширений

In [ ]:
%load_ext tensorboard

### Подключение Google-диска при запуске в Google Colab 

In [ ]:
if platform == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Проверка основных настроек

In [ ]:
assert isinstance(EMOTIONS, (list, tuple)) or isinstance(EMOTIONS, dict), 'Эмоции должны быть списком или кортежем названий эмоций, либо словарем ключами которого являются названия эмоций, а значениями пары value-arousal.'
assert len(EMOTIONS) > 1, 'Количество эмоций должно быть больше 1.'
assert all([isinstance(emotion, str) for emotion in EMOTIONS]), 'Названия эмоций должны быть строками.'
if isinstance(EMOTIONS, dict):
    assert all(isinstance(value, (list, tuple)) for value in EMOTIONS.values()), 'Значения value-arousal должны быть заданы списком или кортежем чисел.'
    assert all((isinstance(x, (int, float)) for x in value) for value in EMOTIONS.values()), 'Значения value-arousal должны быть числами.'
    assert all(len(value)==2 for value in EMOTIONS.values()), 'В списке или кортеже значений value-arousal должны быть два элемента.'

In [ ]:
assert isinstance(PROJECT_NAME, str), 'Название проекта должно быть задано строкой.'
assert PROJECT_NAME != '', 'Название проекта не может быть пустой строкой.'

In [ ]:
if platform == 'colab':
    PROJ_PATH = COLAB_PROJ_PATH
else:
    PROJ_PATH = LOCAL_PROJ_PATH
assert isinstance(PROJ_PATH, str), 'Путь к папке проекта должен быть задан строкой.'
proj_path = Path(PROJ_PATH)
assert Path(proj_path).parent.exists(), 'Отсутствует папка для размещения папки проекта.'

In [ ]:
if platform == 'colab':
    GD_PROJ_PATH = COLAB_GD_PROJ_PATH
else:
    GD_PROJ_PATH = LOCAL_GD_PROJ_PATH
assert isinstance(GD_PROJ_PATH, str), 'Путь к папке проекта на Google-Диске должен быть задан строкой.'
gd_proj_path = Path(GD_PROJ_PATH)
assert Path(gd_proj_path).parent.exists(), 'На Google-Диске отсутствует папка для размещения папки проекта.'

In [ ]:
assert isinstance(TRAIN_DATASET_URL, str), 'Ссылка на архив тренировочного датасета должна быть задана строкой.'
assert validators.url(TRAIN_DATASET_URL), 'Ссылка на архив тренировочного датасета имеет некорректный формат.'

In [ ]:
assert TRAIN_DATASET_EXT in ("zip", "tar", "tar.gz", "tgz", "tar.bz2", "tbz"), \
'Файл архив исходного тренировочного датасета должен иметь любое из следущих расширений: *.zip, *.tar, *.tar.gz, *.tgz, *.tar.bz2, *.tbz'

In [ ]:
assert isinstance(TRAIN_DATASET_PATH, str), 'Путь к исходному тренировочному датасету внутри папки проекта должен быть задана строкой.'
try:
    Path(TRAIN_DATASET_PATH).exists()
    valid = True
except OSError as e:
    valid = False
assert valid, 'Синтаксическая ошибка в пути к исходному тренировочному датасету внутри папки проекта.' 

In [ ]:
assert isinstance(TEST_DATASET_URL, str), 'Ссылка на архив тестового датасета должна быть задана строкой.'
assert validators.url(TEST_DATASET_URL), 'Ссылка на архив тестового датасета имеет некорректный формат.'

In [ ]:
assert TEST_DATASET_EXT in ("zip", "tar", "tar.gz", "tgz", "tar.bz2", "tbz"), \
'Файл архив исходного тестового датасета должен иметь любое из следущих расширений: *.zip, *.tar, *.tar.gz, *.tgz, *.tar.bz2, *.tbz'

In [ ]:
assert isinstance(TEST_DATASET_PATH, str), 'Путь к исходному тестовому датасету внутри папки проекта должен быть задана строкой.'
try:
    Path(TEST_DATASET_PATH).exists()
    valid = True
except OSError as e:
    valid = False
assert valid, 'Синтаксическая ошибка в пути к исходному тестовому датасету внутри папки проекта.' 

In [ ]:
assert isinstance(KAGGLE_API_TOKEN_URL, str), 'Ссылка на токен для подключения к платформе Kaggle через API должна быть задана строкой.'
assert validators.url(KAGGLE_API_TOKEN_URL), 'Ссылка на токен для подключения к платформе Kaggle через API имеет некорректный формат.'

In [ ]:
assert isinstance(MAX_INFERENCE_TIME, (int, float)), 'Максимально допустимое время инференса модели в секундах должно быть числом с плавающей точкой.'
assert MAX_INFERENCE_TIME > 0., 'Максимально допустимое время инференса модели в секундах должно быть положительным числом с плавающей точкой.'

In [ ]:
assert isinstance(INFERENCE_TIME_WEIGHT, (int, float)), 'Вес времени инференса модели при выборе базовой модели должен быть числом.'
assert INFERENCE_TIME_WEIGHT >= 0.0 and INFERENCE_TIME_WEIGHT <= 1.0, 'Вес времени инференса при выборе базовой модели должен находиться в диапазоне [0., 1.].'

In [ ]:
assert isinstance(BASE_MODEL_MAX_SIZE, (int, float)), 'Максимально допустимый размер базовой модели в МБ должен быть числом.'
assert BASE_MODEL_MAX_SIZE > 0.0, 'Максимально допустимый размер базовой модели в МБ должен быть положительным числом.'

In [ ]:
assert BASE_MODEL_POOLINGS in ('avg', 'max'), 'Тип пулинга на выходе базовой модели должен быть либо "avg" (average), либо "max" (max).'

In [ ]:
assert isinstance(MODEL_ON_TOP_DENSE_NUMS, (list, tuple)), 'Варианты количества дополнительных полносвязных слоев должны быть заданы списком или кортежем.'
assert all(isinstance(num, int) for num in MODEL_ON_TOP_DENSE_NUMS), 'Количество дополнительных полносвязных слоев должны быть целым числом.'
assert all(num > 0 for num in MODEL_ON_TOP_DENSE_NUMS), 'Количество дополнительных полносвязных слоев должны быть положительным числом.'
assert len(MODEL_ON_TOP_DENSE_NUMS) >= 1, 'Должен быть задан хотя бы один вариант количества дополнительных полносвязных слоев.'

In [ ]:
assert isinstance(MODEL_ON_TOP_DENSE_UNITS, (list, tuple)), 'Варианты количества выходных нейронов в дополнительном полносвязном слое должны быть заданы списком или кортежем.'
assert all(isinstance(num, int) for num in MODEL_ON_TOP_DENSE_UNITS), 'Количество выходных нейронов в дополнительном полносвязном слое должны быть целым числом.'
assert all(num > 0 for num in MODEL_ON_TOP_DENSE_UNITS), 'Количество выходных нейронов в дополнительном полносвязном слое должны быть положительным числом.'
assert len(MODEL_ON_TOP_DENSE_UNITS) >= 1, 'Должен быть задан хотя бы один вариант количества выходных нейронов в дополнительном полносвязном слое.'

In [ ]:
assert isinstance(MODEL_ON_TOP_DROPOUT_RATES, (list, tuple)), 'Варианты доли исключаемых данных перед подачей в полносвязный слой при обучении должны быть заданы списком или кортежем.'
assert all(isinstance(num, float) for num in MODEL_ON_TOP_DROPOUT_RATES), 'Доли исключаемых данных перед подачей в полносвязный слой при обучении должны быть целым числом.'
assert all((num >= 0. and num < 1.0) for num in MODEL_ON_TOP_DROPOUT_RATES), 'Доли исключаемых данных перед подачей в полносвязный слой при обучении должны быть положительным числом.'
assert len(MODEL_ON_TOP_DROPOUT_RATES) >= 1, 'Должен быть задан хотя бы один вариант доли исключаемых данных перед подачей в полносвязный слой при обучении.'

In [ ]:
assert isinstance(MODEL_ON_TOP_INITIAL_LEARNING_RATE, float), 'Начальная скорость обучения полносвязной модели должна быть числом.'
assert MODEL_ON_TOP_INITIAL_LEARNING_RATE > 0.0, 'Вес времени инференса при выборе базовой модели должна быть положительным числом.'

In [ ]:
assert isinstance(MODEL_ON_TOP_LEARNING_RATE_DECAY_RATE, float), 'Коэффициент изменения скорости обучения полносвязной модели по окончании каждой эпохи должен быть числом.'
assert MODEL_ON_TOP_LEARNING_RATE_DECAY_RATE > 0.0, 'Коэффициент изменения скорости обучения полносвязной модели по окончании каждой эпохи должен быть положительным числом.'

In [ ]:
assert isinstance(MODEL_INITIAL_LEARNING_RATE, float), 'Начальная скорость обучения модели при тонкой настройке должна быть числом.'
assert MODEL_INITIAL_LEARNING_RATE > 0.0, 'Начальная скорость обучения модели при тонкой настройке должна быть положительным числом.'

In [ ]:
assert isinstance(MODEL_LEARNING_RATE_DECAY_RATE, float), 'Коэффициент изменения скорости обучения модели по окончании каждой эпохи при тонкой настройке должен быть числом.'
assert MODEL_LEARNING_RATE_DECAY_RATE > 0.0, 'Коэффициент изменения скорости обучения модели по окончании каждой эпохи при тонкой настройке должен быть положительным числом.'

In [ ]:
assert hasattr(optimizers, OPTIMIZER), f'Оптимизатор {OPTIMIZER} отсутствует в библиотеке Keras.'

In [ ]:
assert VERBOSE in (0, 1), f'Некорреткно указан режим верболизации {VERBOSE}. Значение должен быть равно 0 (тихий) или 1 (вывод сообщений).'

In [ ]:
assert isinstance(SEED, int) or SEED is None, 'Инициализатор генератора случайных чисел должен быть целым числом или None.'

### Подготовка к первому запуску

#### Создание папки проекта

In [ ]:
if not proj_path.exists():
    proj_path.mkdir()

#### Создание папки проекта на Google Диск

In [ ]:
if not gd_proj_path.exists():
    gd_proj_path.mkdir()

#### Копирование токена kaggle

In [ ]:
if platform == 'colab':
    kaggle_path = Path('/root/.kaggle/kaggle.json')
else:
    kaggle_path = Path('.kaggle/kaggle.json')
if not kaggle_path.parent.exists():
    kaggle_path.parent.mkdir()
if not kaggle_path.exists():
    gdown.download(KAGGLE_API_TOKEN_URL, kaggle_path.as_posix(), fuzzy=True)

### Установка папки проекта в качестве рабочей директории

In [ ]:
%cd {proj_path}

/content/skillbox-computer-vision-project


### Вывод информации о выделенном GPU и доступной виртуальной памяти в Google Colab

In [ ]:
if platform == 'colab':
    gpu_info = !nvidia-smi
    gpu_info = '\n'.join(gpu_info)
    if gpu_info.find('failed') >= 0:
        print('Not connected to a GPU')
    else:
        print(gpu_info)
    ram_gb = virtual_memory().total / 1.024e9
    print(f'\nYour runtime has {ram_gb:.1f} gigabytes of available RAM')

Not connected to a GPU

Your runtime has 13.3 gigabytes of available RAM


## Вспомогательные функции и классы

### Построение и обучение моделей

#### Построение модели аугментации

In [ ]:
def build_augment_model(image_size: int, 
                        flip: Optional[str]=None, 
                        rotation_factor: Optional[float]=None, 
                        zoom_factor: Optional[float]=None, 
                        contrast_factor: Optional[float]=None, 
                        brightness_factor: Optional[float]=None, 
                        training: bool=False, 
                        seed=SEED) -> models.Model:
    '''Создает модель аугментации входного изображения квадратной формы. Аугментация достигается за счёт случайного зеркалирования, поворота, масштабирования, изменения контраста и яркости исходного изображения. 
    Модель работает только в режиме инференса и только при обучении, при условии, что установлен флаг training.
    
    Аргументы:
    - image_size: размер изображения,
    - flip: тип зеркалирования: 'horizonal' (горизонтальный), 'vertical' (вертикальный), если None, то зеркалирование не производится.
    - rotation_factor: величина угол повотора (по или против часовой стрелки) изображения в долях от полного оборота, если None, то поворот не производится.
    - zoom_factor: величина максимального изменения (увеличения или уменьшения) изображения в долях от исходного размера, если None, то масштаб не меняется.
    - contrast_factor: величина максимального изменения (увеличения или уменьшения) контраста изображения в долях от исходного контраста, если None, то контраст не меняется.
    - brightness_factor: величина максимального изменения (увеличения или уменьшения) яркости изображения в долях от исходной яркости, если None, то яркость не меняется.
    - training: если False, то модель не работает, если True, то модель работает во время обучения.
    - seed: инициализатор генератора случайных чисел.'''
    i = layers.Input(shape=(image_size, image_size, 3), name='original_image_input')
    x = i
    if flip is not None:
        x = layers.RandomFlip(flip, seed=seed, name='random_flip')(x, training=training)
    if rotation_factor is not None:
        x = layers.RandomRotation(rotation_factor, seed=seed, name='random_rotation')(x, training=training)
    if zoom_factor is not None:
        x = layers.RandomZoom(zoom_factor, seed=seed, name='random_zoom')(x, training=training)
    if contrast_factor is not None:
        x = layers.RandomContrast(contrast_factor, seed=seed, name='random_contrast')(x, training=training)
    if brightness_factor is not None:
        x = layers.RandomBrightness(brightness_factor, seed=seed, name='random_brightness')(x, training=training)
    o = x
    model = models.Model(inputs=[i], outputs=[o], name='augmentation_model')
    return model

#### Построение базовой модели

In [ ]:
def build_base_model(name: str, 
                     weights: Optional[str]='imagenet', 
                     image_size: Optional[int]=None, 
                     pooling: str='avg', 
                     include_preprocess_input:bool=False, 
                     training: bool=False) -> models.Model:
    '''Создает базовую модель из библиотеки Keras Applications. Модель обрабатывает изображение квадратной формы.
    
    Аргументы:
    - name: название базовой модели.
    - weights: начальные веса модели: None (случайные), 'imagenet' (полученные при обучении на датасете ImageNet) или путь к файлу с весами.
    - image_size: размер входного изображения.
    - pooling: тип пулинга выходного слоя: 'avg' (average) или 'max' (max).
    - include_preprocess_input: добавление слоев предобработки изображений.
    - training: использование модели для обучения.'''

    # Форму входного изображения указываем, только если задан размер входного изображения
    input_shape = (image_size, image_size, 3)

    # Создаем ядро базовой модели
    bild_core = getattr(applications, name)
    core = bild_core(include_top=False, weights=weights, input_shape=input_shape, pooling=pooling)

    # Создаем входной слой для изображения в виде массива uint8
    i = layers.Input(input_shape, name=f'image_input')

    # Добавляем предварительную обработку
    if include_preprocess_input:
        app_module = inspect.getmodule(bild_core)
        build_preprocess_input = getattr(app_module, 'preprocess_input')
        x = build_preprocess_input(i)
    else:
        x = i

    # Добавляем базовую модель
    o = core(x, training=training)
    
    # Соединяем все в одну модель
    model = models.Model(inputs=[i], outputs=[o], name=core.name + ('_with' if include_preprocess_input else '_without') + '_preprocessing')

    # Возвращаем полученную модель
    return model

#### Построение полносвязной модели

In [ ]:
def build_model_on_top(
    feature_size: int,
    config: Union[List[Tuple[float, int]], Tuple[Tuple[float, int]]],
    emotions: Union[Union[List[str], Tuple[str]], Dict[str, Tuple[float, float]]]=EMOTIONS,
    training: bool=False,
    seed: int=SEED,
) -> models.Model:
    '''Создает модель из полносвязных слоев. Для регуляризации при обучении используется метод 'Dropout'. 
    В случае если модель предсказывает эмоции, количество выходных нейронов в выходном соответствует количеству эмоций и применяется активация типа 'Softmax'.
    В случае если модель предсказывает valance-arousal эмоций, то в выходном слое только 2 выходных нейрона и применяется активация тип 'ReLU' с ограничением выходных значений до 2.
    Для приедения выходных значений valance-arousal добавляется последний слой, обеспечивающий уменьшений значений valance-arousal на 1.
    
    Аргументы:
    - feature_size: размер признаков.
    - config: конфигурация слоев регуляризации и полносвязных слоев (за исключением выходного) в формате: [(коэффициент регуляризации, размер полносвязного слоя), ...]
    - emotions: описание предсказываемых эмоций.
    - training: для тренировки.
    - seed: инициализатор генератора случайных чисел слоев регуляризации.'''   
    
    # Создаем входной слой модели
    i = layers.Input(shape=(feature_size,), name='feature_input')
    x = i
    
    # Инициализация ядра
    initializer = initializers.GlorotUniform(seed=seed)

    # Добавляем полносвязные слои в соответствии с конфигурацией
    for index, (dropout_rate, dense_units) in enumerate(config):
        if dropout_rate > 0.:
            x = layers.Dropout(dropout_rate, seed=seed, name=f'dropout_{index}')(x, training=training)
        if dense_units > 0:
            x = layers.Dense(dense_units, kernel_initializer=initializer, activation="relu", name=f'dense_{index}')(x, training=training)
    
    # Добавляем выходной полносвязный слой
    if isinstance(emotions, (list, tuple)):
        # При обычной классификации
        o = layers.Dense(len(emotions), kernel_initializer=initializer, activation="softmax", name='probs')(x, training=training)
    else:
        # При использовании разложения эмоций "valence-arousal"
        x = layers.Dense(2, kernel_initializer=initializer, name='dense_valence_arousal')(x, training=training)
        x = layers.ReLU(max_value=2.0)(x)
        o = layers.Lambda(lambda x: x - 1.0)(x)

    # Создаем модель
    model = models.Model(inputs=i, outputs=o, name='model_on_top')
    
    return model

#### Построение модели

In [ ]:
def build_model(
    augment_model: Optional[models.Model]=None, 
    base_model: Optional[models.Model]=None, 
    model_on_top: Optional[models.Model]=None,
) -> models.Model:
    '''Создает модель предсказания эмоций из трех моделей, последовательно соединяемых модей: модели аугментации изображений, базовой модели и полносвязной модели.
    
    Аргументы:
    - augment_model: модель аугментации изображений (см. функцию build_augment_model).
    - base_model: базовая модель (см. функцию build_base_model).
    - model_on_top: полносвязная модель (см. функцию build_on_top_model).
    '''
    models_list = []
    if augment_model is not None:
        models_list.append(augment_model)
    if base_model is not None:
        models_list.append(base_model)
    if model_on_top is not None:
        models_list.append(model_on_top)
    if len(models_list) == 0:
        return
    elif len(models_list) == 1:
        model = models_list[0]
    else:
        model = models.Sequential(models_list)
    return model

#### Остановка обучения

In [ ]:
class EarlyStoppingAtMaxTestScore(callbacks.Callback):
    """Останавливает обучение модели обучения эмоций, когда оценка предсказания модели на тестовых данных на платформе Kaggle перестает расти."""

    def __init__(
        self, 
        model: models.Model,
        test_dataset: tf.data.Dataset, 
        test_image_paths: str, 
        patience: int=0, 
        restore_best_weights: bool=False,
        verbose: int=VERBOSE, 
        emotions: Union[Union[List[str], Tuple[str]], Dict[str, Tuple[float, float]]]=EMOTIONS,
        best_epoch: int=-1, 
        best_test_score: float=0., 
        best_weights: Optional[List[np.array]]=None,
        wait: int=0
    ):
        '''Аргументы:

        - model: обучаемая модель
        - test_dataset: датасет тестовых изображений.
        - test_image_paths: список путей к файлам тестового датасета.
        - patience: количество эпох, если в течение которых оценка не улучшается, то обучение останавливается.
        - verboes: режим верболизации работы: 0 (тихий) или 1 (вывод сообщений). 
        - best_epoch: номер эпохи обучения с лучшей оценкой (по окончании предыдушей итерации).
        - best_test_score: лучшая оценка (по окончании предыдушей итерации).
        - best_weights: веса по окончании эпохи с лучшей оценкой модели (по окончании предыдушей итерации).
        - wait: количество эпох, в течение которых оценка не улушалась (по окончании предыдушей итерации).'''
    
        super(EarlyStoppingAtMaxTestScore, self).__init__()

        self.__model = model
        self.__emotions = emotions
        
        self.__description = f'test_{model.name}'
        self.__file_path = f'test_{model.name}.csv'
        self.__keras = Kaggle()
        
        self.__test_dataset = test_dataset
        self.__test_result = pd.DataFrame(columns=['image_path', 'emotion'])
        self.__test_result['image_path'] = test_image_paths
        
        self.__restore_best_weights = restore_best_weights
        self.__patience = patience
        self.__verbose = verbose
        self.__best_epoch = best_epoch
        self.__best_test_score = best_test_score
        self.__best_weights = best_weights
        self.__wait = wait
        
    @property
    def best_epoch(self):
        return self.__best_epoch

    @property
    def best_test_score(self):
        return self.__best_test_score
    
    @property
    def best_weights(self):
        return self.__best_weights

    def on_train_begin(self, logs=None):
        self.__stopped_epoch = -1

    def on_epoch_end(self, epoch, logs=None):
        # Окончание эпохи обучения
        
        # Проверяем точность предсказаний модели в Kaggle
        predicts = self.__model.predict(test_dataset, verbose=self.__verbose)
        if isinstance(EMOTIONS, (list, tuple)):
            labels = predicts.argmax(axis=1).tolist()
            self.__test_result['emotion'] = [self.__emotions[label] for label in labels]
        else:
            dists = np.apply_along_axis(lambda a: np.linalg.norm(a - np.array(list(self.__emotions.values())), axis=1), arr=predicts, axis=1)
            labels = dists.argmin(axis=1).tolist()
            self.__test_result['emotion'] = [list(self.__emotions)[label] for label in labels]
        self.__test_result.to_csv(self.__file_path, index=False)
        self.__keras.send_submission_files(descriptions=[self.__description], file_paths=[self.__file_path])
        Path(self.__file_path).unlink()
        test_scores = self.__keras.receive_submission_scores(descriptions=[self.__description]).loc[0, ['publicScore', 'privateScore']]
        current = (test_scores['publicScore'] + test_scores['privateScore']) / 2
        logs['test_public_score'] = test_scores['publicScore']
        logs['test_private_score'] = test_scores['privateScore']
        logs['test_score'] = current
        
        if current > self.__best_test_score:
            # Точность улучшилась - запонимаем значение и веса модели
            self.__best_test_score = current
            self.__best_epoch = epoch
            self.__wait = 0
            self.__best_weights = self.__model.get_weights()
            if self.__verbose:
                print(f'Epoch #{self.__best_epoch + 1}: accuracy has been improved ({current:.4f}).')
        else:
            # Точность не улучшилась - ждем в течении заданного количества эпох, потом останаваливаем обучение
            self.__wait += 1
            if self.__wait >= self.__patience:
                self.__stopped_epoch = epoch
                self.__model.stop_training = True

    def on_train_end(self, logs=None):
        # Остановка обучения
        if self.__stopped_epoch >= 0:
            if self.__verbose:
                print(f"Epoch #{self.__stopped_epoch + 1}: early stopping.")
        
        # Восстановление весов лучшей эпохи
        if self.__restore_best_weights:
            if self.__verbose:
                print(f"Restoring model weights from the end of the best epoch (#{self.__best_epoch + 1}).")
            self.__model.set_weights(self.__best_weights)

#### Экспоненциальное снижение скорости обучения

In [ ]:
class LearningRateExpDecayScheduler(callbacks.LearningRateScheduler):
    '''Экпоненциальное снижение скорости обучения.'''
    def __init__(self, decay_rate: float=1., verbose: int=VERBOSE):
        '''Аргументы:
        - decay_rate - коэффициент изменения [0.0, 1.0].
        - verbose - режим верболизации: 0-тихий, 1-вывод сообщений об изменении скорости обучения.'''
        self.__decay_rate = decay_rate
        super(LearningRateExpDecayScheduler, self).__init__(self.__scheduler)
        
    def __scheduler(self, epoch, lr) -> float:
        return lr * self.__decay_rate

#### Модель для внедрения

In [ ]:
class FaceEmotionRecognitionNet():
    
    def __init__(self, file_path: str, emotions: Union[Union[List[str], Tuple[str]], Dict[str, Tuple[float, float]]]=EMOTIONS):
        '''
        file_path: путь к файлу сохраненной модели.
        emotions: предсказываемые эмоции.
        '''
        # Загружаем модель
        self.__model = models.load_model(filepath=file_path, compile=False)
        self.__emotions = emotions
        
    def predict(self, face_image: np.array) -> Union[Tuple[str, float], Tuple[str, float, float, float]]:
        '''Предсказание эмоции человека по изображению его лица.
        
        Аргументы:
        - face_image: изображение лица человека.
        '''
        image = Image.fromarray(face_image)
        size = max(image.width, image.height)
        # Делаем изображение квардратным
        padded_image = ImageOps.pad(image, (size, size))
        # Подгоняем размер изображения 
        resized_image = padded_image.resize(self.__model.input_shape[1:3])
        # Получаем предсказание
        tensor = np.asarray(resized_image)[None, ...]
        predicts = self.__model.predict(tensor, verbose=0)[0]
        # Готовим результирующие данные
        if isinstance(self.__emotions, list):
            probability = predicts.max()
            label = predicts.argmax()
        else:
            valence, arousal = predicts
            dists = np.apply_along_axis(lambda a: np.linalg.norm(a - np.array(list(self.__emotions.values())), axis=1), arr=predicts[None, ...], axis=1)
            error = dists.min()
            label = dists.argmin()
        emotion = self.__names[label]
        # Возвращаем результат
        if isinstance(self.__emotions, list):
            return emotion, probability
        else:
            return emotion, error, valence, arousal

### Датасет

#### Функция извлечения лица из батча изображений с сохранением в файлы

In [ ]:
def extract_faces(dataset_path: Path, faces_dataset_path: Path, image_rel_paths: Union[List[str], Tuple[str]], params: dict, skip_failed_image: bool=True):
    '''Извлечает лица из изображений.
    
    Аргументы:
    dataset_path - путь к датасету исходных изображений.
    faces_dataset_path - путь к датасету извлечённых изображений лиц.
    image_rel_paths - список путей к файлам обрабатываемых изображений.
    skip_failed_image - пропускать или нет изображение, на котором не удалось найти ни одного лица.'''
    
    import numpy as np
    from PIL import Image, ImageOps
    from mtcnn.mtcnn import MTCNN
    
    # Создаем детектор лиц
    detector = MTCNN(scale_factor=params['scale_factor'], min_face_size=params['min_face_size'])
    
    result = []
    
    for image_rel_path in image_rel_paths:
    
        # Загружаем исходное изображение
        with Image.open(dataset_path / image_rel_path) as image:

            # Находим лица на изображении
            faces = detector.detect_faces(np.asarray(image))

            # Если детектор не нашел изображение лица,
            # то прекращаем обработку изображения
            if len(faces) == 0:
                size = max(image.width, image.height)
                if skip_failed_image:
                    result.append((size, 0, 0))
                else:
                    # Делаем извлеченное изображение квадратным,
                    # добавление "пустых" областей
                    padded_image = ImageOps.pad(image, (size, size))
                    # Сохраняем изображение лица в папку датасета
                    padded_image.save(faces_dataset_path / image_rel_path)
                    # Возвращаем размеры начальный и конечный
                    result.append((size, 1, size))
                continue
                    
            # Извлекаем изображение первого найденного лица
            x, y, w, h = faces[0]['box']
            w = min(w, image.width - x)
            h = min(h, image.height - y)
            box = (x, y, image.width-x-w, image.height-y-h)
            face_image = ImageOps.crop(image, box)

            # Делаем извлеченное изображение квадратным,
            # добавление "пустых" областей
            size = max(face_image.width, face_image.height)
            face_padded_image = ImageOps.pad(face_image, (size, size))

            # Сохраняем изображение лица в папку датасета
            face_padded_image.save(faces_dataset_path / image_rel_path)

            # Возвращаем размеры начальный и конечный
            result.append((max(image.width, image.height), len(faces), size))
            
    return result

#### Функция создания датасета признаков

In [ ]:
def build_feature_dataset(
    file_path: str, 
    emotions: Union[Union[List[str], Tuple[str]], Dict[str, Tuple[float, float]]]=EMOTIONS,
    labeled: bool=True, 
    batch_size: int=1, 
    shuffle: bool=True, 
    reshuffle_each_iteration: bool=True, 
    seed: int=SEED, 
    validation_split: Optional[float]=None, 
    test_split: Optional[float]=None
) -> Union[
    tf.data.Dataset, 
    Tuple[tf.data.Dataset, tf.data.Dataset], 
    Tuple[tf.data.Dataset, tf.data.Dataset, tf.data.Dataset]
]:
    '''Загружает датасет признаков изображений из файла массивов. 
    Возвращает датасет, разделенный на части: тренировочную, проверочную (опционально) и тестовую (опционально).
    Опционально данные могут быть перемещаны случайным образом.
    Датасет по-умолчанию содержит разметку, но опционально она может быть исключена из него.
    Датасет(-ы) разделяются на батчи заданного размера.
    
    Аргументы:
    - file_path: путь к файлу данных.
    - emotions: описание эмоций в датасете.
    - batch_size: размер батча.
    - shuffle: перемешивание данных.
    - reshuffle_each_iteration: перешивание данных при каждой итерации.
    - seed: инициализатор генератора случайных чисел при перемешивании.'''

    # Создаем тренировочный и валидационный датасеты
    with np.load(file_path, allow_pickle=True) as data:
        if not labeled:
            dataset = tf.data.Dataset.from_tensor_slices(data['features'])
        elif isinstance(EMOTIONS, (list, tuple)):
            dataset = tf.data.Dataset.from_tensor_slices((data['features'], data['labels']))
        else:
            labels = np.apply_along_axis(lambda label: list(EMOTIONS.values())[int(label)], axis=1, arr=data['labels'])
            dataset = tf.data.Dataset.from_tensor_slices((data['features'], labels))
    size = len(dataset)

    # Перемешиваение
    if shuffle:
        dataset = dataset.shuffle(size, seed, reshuffle_each_iteration=reshuffle_each_iteration)
        
    # Возвращение полного датасета
    if validation_split is None and test_split is None:
        return dataset.batch(batch_size)
    
    # Возвращение датасета, разделенного на тренировочную и проверочную части
    if test_split is None:
        train_size = int(size * (1-validation_split))
        val_size = size - train_size
        train_dataset = dataset.take(train_size).batch(batch_size)
        val_dataset = dataset.skip(train_size).take(val_size).batch(batch_size)
        return train_dataset, val_dataset
    
    # Возвращение датасета, разделенного на тренировочную, проверочную и тестовую части
    train_size = int(size * (1-validation_split-test_split))
    val_size = int(size * validation_split)
    test_size = size - train_size - val_size
    train_dataset = dataset.take(train_size).batch(batch_size)
    val_dataset = dataset.skip(train_size).take(val_size).batch(batch_size)
    test_dataset = dataset.skip(train_size + val_size).take(test_size).batch(batch_size)
    return train_dataset, val_dataset, test_dataset

### Пайплайн

In [ ]:
class Pipeline():
    '''Пайплан создания модели.'''
    
    def __init__(self, config: dict, proj_path: Path, is_prev_complete: bool, platform: str):
        '''Аргументы:
        - config[dict]: конфигурация этапа.
        - proj_path[Path]: путь к папке проекта.
        - is_prev_complete[bool]: выполнен ли предыдущий этап?
        - platform[str]: платформа, на которой выполяется пайплайн: 'colab' (Goggle Colab), 'local' (локальная).
        '''
        self.__name = config['name']
        self.__description = config['description']
        self.__stages = (stage for stage in config['stages'])
        self.__path = proj_path / config['name']
        if not self.__path.exists():
            self.__path.mkdir()
        self.__report_path = self.__path / config['report_csv']
        if self.__report_path.exists():
            self.__report = pd.read_csv(self.__report_path, index_col='stage')
        else:
            self.__report = pd.DataFrame(
                columns = [
                    'stage',
                    'params',
                    'platform',
                    'start_time',
                    'update_time',
                    'state',
                ]
            )
            self.__report['stage'] = [stage['name'] for stage in config['stages']]
            self.__report.set_index('stage', inplace=True)
        self.__is_prev_complete = is_prev_complete
        self.__platform = platform
        self.__stage = None
            
    def next_stage(self):
        '''Возвращает следующий этап пайплайна.'''
        self.__stage = next(self.__stages)
        
        # Если этап уже выполнен, то пропускаем его выполнение
        if self.__report.loc[self.__stage['name'], 'state'] == 'complete':
            return
        
        # Этап ещё не выполнен
        self.__report.loc[self.__stage['name'], 'platform'] = self.__platform
        self.__report.loc[self.__stage['name'], 'params'] = str(self.__stage['params'])
        
        # Если предыдущий пайплайн ещё не выполнен полностью, или ещё не выполнены все предыдущие этапы этого пайплайна 
        # то пропускаем этап
        if not self.__is_prev_complete or (self.__report.iloc[:self.__report.index.get_loc(self.__stage['name'])]['state'] != 'complete').any():
            if self.__report.loc[self.__stage['name'], 'state'] != 'skipped (not ready)':
                self.__report.loc[self.__stage['name'], 'update_time'] = datetime.now()
                self.__report.loc[self.__stage['name'], 'state'] = 'skipped (not ready)'
            return
        
        # Если рантайм не соответствует требуемому, то пропускаем этап
        if self.__platform != self.__stage['platform']:
            if self.__report.loc[self.__stage['name'], 'state'] != 'skipped (platform)':
                self.__report.loc[self.__stage['name'], 'update_time'] = datetime.now()
                self.__report.loc[self.__stage['name'], 'state'] = 'skipped (platform)'
            return
        
        # Если этап выполняется в несколько итераций, то переходим к очередной итерации
        if self.__report.loc[self.__stage['name'], 'state'] == 'run complete':
            self.__report.loc[self.__stage['name'], 'update_time'] = datetime.now()
            self.__report.loc[self.__stage['name'], 'state'] = 'run started'
            return
        
        # Запоминаем время начала выполнения этапа
        self.__report.loc[self.__stage['name'], 'start_time'] = datetime.now()
        self.__report.loc[self.__stage['name'], 'update_time'] = self.__report.loc[self.__stage['name'], 'start_time']
        self.__report.loc[self.__stage['name'], 'state'] = 'started'
    
    @property
    def name(self) -> str:
        '''Название пайплайна.'''
        return self.__name
        
    @property
    def description(self) -> str:
        '''Описание пайплайна.'''
        return self.__description
        
    @property
    def report(self) -> pd.DataFrame:
        '''Отчёт о выполнении пайплайна.'''
        return self.__report
        
    @property
    def stage_name(self) -> str:
        '''Название текущего этапа.'''
        return self.__stage['name']
        
    @property
    def is_complete(self) -> bool:
        '''Выполнен ли текущий пайплайн?'''
        return (self.__report['state'] == 'complete').all()
        
    @property
    def is_stage_failed(self) -> bool:
        return self.__report.loc[self.__stage['name'], 'state'] == 'failed'
        
    @property
    def is_stage_complete(self) -> bool:
        '''Выполнен ли текущий этап?'''
        return self.__report.loc[self.__stage['name'], 'state'] == 'complete'
        
    @property
    def is_stage_started(self) -> bool:
        '''Выполняется ли текущий этап?'''
        return self.__report.loc[self.__stage['name'], 'state'] == 'started'

    @property
    def is_stage_skipped(self) -> bool:
        '''Пропускается ли текущий этап?'''
        return not self.__report.loc[self.__stage['name'], 'state'].find('started') >= 0

    @property
    def stage_params(self) -> dict:
        '''Параметры текущего этапа.'''
        return self.__stage['params']
        
    @property
    def stage_description(self) -> str:
        '''Описание этапа.'''
        return self.__stage['description']

    def __save(self):
        '''Сохранение пайплана в файл.'''
        self.__report.to_csv(self.__report_path)
        
    def complete_stage_run(self):
        '''Окончание итерации этапа.'''
        self.__report.loc[self.__stage['name'], 'state'] = 'run complete'
        self.__save()
        
    def fail_stage(self):
        self.__report.loc[self.__stage['name'], 'update_time'] = datetime.now()
        self.__report.loc[self.__stage['name'], 'state'] = 'failed'
        self.__save()
        
    def complete_stage(self):
        '''Окончание этапа.'''
        self.__report.loc[self.__stage['name'], 'update_time'] = datetime.now()
        self.__report.loc[self.__stage['name'], 'state'] = 'complete'
        self.__save()
        
    def save_stage_processing(self, result: pd.DataFrame):
        '''Сохранение журнала выполнения этапа в файл.'''
        result.to_csv(self.__path / self.__stage['params']['process_csv'])
        
    def load_stage_processing(self) -> pd.DataFrame:
        '''Загрузка журнала выполнения этапа из файла.'''
        return pd.read_csv(self.__path / self.__stage['params']['process_csv'])
    
    def save_stage_result(self, result: pd.DataFrame):
        '''Сохранение результата выполнения этапа в файл.'''
        result.to_csv(self.__path / self.__stage['params']['result_csv'])
        
    def load_stage_result(self) -> pd.DataFrame:
        '''Загрузка результата выполнения этапа из файла.'''
        return pd.read_csv(self.__path / self.__stage['params']['result_csv'])

### Kaggle

In [ ]:
class Kaggle():
    '''Взаимодействие с платформой Kaggle.'''
    
    __COLUMNS = ['fileName', 'date', 'description', 'status', 'publicScore', 'privateScore']
            
    def __init__(self, competition: str=PROJECT_NAME, verbose: int=VERBOSE):
        '''Инициализация взаимодействия с с платформой Kaggle.
        
        Аргументы:
        - competition: название соревнования.
        - verbose: режим верболизации: 0 (тихий) или 1 (вывод сообщений).'''
        
        self.__competition = competition
        self.__verbose = verbose
        
    def send_submission_files(self, descriptions: Union[List[str], Tuple[str]], file_paths: Union[List[str], Tuple[str]]):
        '''Отправка файлов решений на проверку.
        
        Аргументы:
        - descriptions: список описаний решений.
        - file_paths: список путей к файлам решений.'''
        for file_path, description in zip(file_paths, descriptions):
            cmd = f'kaggle competitions submit -c {self.__competition} -f "{file_path}" -m "{description}" -q'
            lines = !{cmd}
            if self.__verbose:
                print(f'Sended file {file_path} of submission {description} to competition {self.__competition}.')
        
    def receive_submission_scores(self, descriptions: Union[List[str], Tuple[str]]) -> pd.DataFrame:
        '''Прием результатов проверки решений.
        
        Аргументы:
        - descriptions: список описаний решений.'''
        
        # Запрашиваем список результатов
        cmd = f'kaggle competitions submissions -c {self.__competition}'
        while True:
            lines = !{cmd}
            if all([line.find('pending') == -1 for line in lines]):
                break
            sleep(1)
        
        if self.__verbose:
            descriptions_str = ', '.join(descriptions)
            print(f'Received scores of submissions {descriptions_str} from competition {self.__competition}.')
        
        # Находим строку заголовков
        for index, line in enumerate(lines):
            if line.split() == self.__COLUMNS:
                break
        
        # Находим положение колонок в тексте
        header_start_positions = [line.find(column) for column in self.__COLUMNS]
        header_end_positions = header_start_positions[1:]
        header_end_positions.append(len(line))
        
        # Оставляем строки с результатами
        lines = lines[index+2:]
        
        # Извлекаем данные из строк результатов
        data = [
            [
                line[header_start_position: header_end_position].strip() 
                for header_start_position, header_end_position in zip(header_start_positions, header_end_positions)
            ] for line in lines
        ]
        
        # Создаем датасет из полученных результатов
        result = pd.DataFrame(data, columns=self.__COLUMNS)
        result['publicScore'] = pd.to_numeric(result['publicScore'], errors='coerce')
        result['privateScore'] = pd.to_numeric(result['privateScore'], errors='coerce')
        result['date'] = pd.to_datetime(result['date'])
        
        # Оставляем только результаты отправленных предсказаний
        result = result[result['description'].isin(descriptions)]
        
        # Т.к. могут быть одноименные файлы, то берем самые поздние
        indexes = sorted([indexes[0] for indexes in result.groupby('description').groups.values()])
        result = result.iloc[indexes]
        
        # Сортируем результаты в порядке отправки
        result = result.sort_values('date').reset_index(drop=True)
        
        # Выводим результаты
        if self.__verbose:
            print(result)
        
        # Возвращаем результат
        return result

## Пайплан предобработки изображений

### Создание/загрузка пайплайна из Google Диск

In [ ]:
pipeline = Pipeline(config=IMAGE_PREPROCESSING_PIPELINE, proj_path=gd_proj_path, is_prev_complete=True, platform=platform)

### Извлечение изображений лиц из тренировочного датасета

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params
train_faces_dataset_path = Path(params['path'])

#### Создание структуры папок датасета

In [ ]:
if not skip:
    
    if train_faces_dataset_path.exists():
        shutil.rmtree(train_faces_dataset_path)
    train_faces_dataset_path.mkdir()
    for emotion in list(EMOTIONS):
        (train_faces_dataset_path / emotion).mkdir()

#### Загрузка исходного тренировочного датасета

In [ ]:
if not skip:
    
    train_dataset_path = Path(TRAIN_DATASET_PATH)
    if not train_dataset_path.exists():
        gdown.cached_download(
            url=TRAIN_DATASET_URL, 
            path=f'temp.{TRAIN_DATASET_EXT}',
            postprocess=gdown.extractall, 
            fuzzy=True)
        Path(f'temp.{TRAIN_DATASET_EXT}').unlink()

#### Получение списка файлов изображений в тренировочном датасете

In [ ]:
if not skip:
    
    file_paths = [Path(file_path).relative_to(train_dataset_path).as_posix() 
                  for file_path in utils.image_dataset_from_directory(train_dataset_path, shuffle=False, batch_size=1).file_paths]

#### Создание таблицы процесса

In [ ]:
if not skip:
    
    processing = pd.DataFrame(
        columns = ['file_path', 'image_size', 'faces_num', 'face_size', 'batch', 'iter', 'engine']
    )
    processing['file_path'] = file_paths
    processing['faces_num'] = 0
    processing['image_size'] = 0
    processing['face_size'] = 0
    processing['batch'] = processing.index // params['batch_size']
    processing['iter'] = processing['batch'] // params['engines']
    processing['engine'] = processing['batch'] % params['engines']

#### Создание таблицы результатов

In [ ]:
if not skip:
    
    train_face_extraction = pd.DataFrame(
        columns = [
            'emotion',
            'failed_images_number',
            'faces_num',
        ],
    )
    train_face_extraction['emotion'] = list(EMOTIONS)
    train_face_extraction.set_index('emotion', inplace=True)

#### Инициализация параллельных вычислений

In [ ]:
if not skip:
    
    # Создаем кластер
    cluster = ipp.Cluster()
    # Запускаем контроллер
    await cluster.start_controller()
    # Запускаем движки по размеру батча
    engine_set_id = await cluster.start_engines(n=params['engines'])
    # Создаем клиента
    rc = await cluster.connect_client()
    # Подключаем клиента к движкам
    rc.wait_for_engines(params['engines'])
    # Создаем наблюдателя
    view = rc.load_balanced_view()

#### Очистка тренировочного датасета с извлечением лиц на изображениях

In [ ]:
if not skip:
    
    with trange(processing.shape[0], unit='file') as t:
        # Идем по итерациям аугментаций
        for iter_index, iter_batches in processing.groupby('iter'):
            # Создаем список результатов выполнения аугментации "движками"
            async_results = []
            # Запускаем аугментацию батча файлов на каждом "движке"
            for engine_index, engine_batch in iter_batches.groupby('engine'):
                # Получаем батч файлов
                file_paths_batch = engine_batch['file_path'].to_list()
                # Запускаем аугментацию батча файлов
                async_result = view.apply_async(extract_faces, train_dataset_path, train_faces_dataset_path, file_paths_batch, params)
                async_results.append(async_result)
            # Ждем окончания обработки батчей всеми "движками"
            rc.wait(async_results)
            # Вносим результат в таблицу процесса
            for engine_index, engine_batch in iter_batches.groupby('engine'):
                processing.loc[engine_batch.index, 'status'] = async_results[engine_index].status
                processing.loc[engine_batch.index, ['image_size', 'faces_num', 'face_size']] = async_results[engine_index].result()
                if async_results[engine_index].status == 'error':
                    # При появлении ошибки завершаем обработку
                    pipeline.fail_stage()
                    break
            # Обновляем счётчик Progress Bar
            t.update(iter_batches.shape[0])

            # Завершение при появлении ошибки
            if pipeline.is_stage_failed:
                break

#### Остановка параллельных вычислений

In [ ]:
if not skip:
    
    await cluster.stop_engines(engine_set_id)
    await cluster.stop_cluster()

#### Сохранение изображений лиц тренировочного датасета в архиве на Google Диск

In [ ]:
if not skip:
    
    if not pipeline.is_stage_failed:
        shutil.make_archive(gd_proj_path / train_faces_dataset_path.name, 'zip', train_faces_dataset_path)

#### Сохранение результатов на Google Диск

In [ ]:
if not skip:
    
    pipeline.save_stage_processing(processing)
    if not pipeline.is_stage_failed:
        for emotion in EMOTIONS.keys():
            indecies = processing.index[processing['file_path'].str.contains(emotion)]
            images_num = indecies.size
            faces_num = sum(processing.loc[indecies, 'faces_num'] > 0)
            failed_num = images_num - faces_num
            train_face_extraction.loc[emotion] = failed_num, faces_num
        train_face_extraction = train_face_extraction.astype(int)
        pipeline.save_stage_result(train_face_extraction)

#### Фиксация завершения этапа

In [ ]:
if not skip and not pipeline.is_stage_failed:
    
    pipeline.complete_stage()

#### Загрузка результатов из Google Диск (выполняется при пропуске этапа)

In [ ]:
if skip:
    
    if pipeline.is_stage_complete:
        train_face_extraction = pipeline.load_stage_result().set_index('emotion')

#### Вывод результатов

In [ ]:
if pipeline.is_stage_complete:
    display(train_face_extraction)

,failed_images_number,faces_num
emotion,,
anger,42,6981
contempt,15,3070
disgust,18,3137
fear,34,5010
happy,29,5926
neutral,53,6742
sad,35,6705
surprise,35,6288
uncertain,50,5877


### Извлечение изображений лиц из тестового датасета

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params
test_faces_dataset_path = Path(params['path'])

#### Загрузка исходного тестового датасета

In [ ]:
if not skip:
    
    test_dataset_path = Path(TEST_DATASET_PATH)
    if not test_dataset_path.exists():
        gdown.cached_download(
            url=TEST_DATASET_URL, 
            path=f'temp.{TEST_DATASET_EXT}',
            postprocess=gdown.extractall, 
            fuzzy=True)
        Path(f'temp.{TEST_DATASET_EXT}').unlink()

#### Создание папки датасета

In [ ]:
if not skip:
    
    if test_faces_dataset_path.exists():
        shutil.rmtree(test_faces_dataset_path)
    test_faces_dataset_path.mkdir()

#### Получение списка файлов изображений в исходном датасете

In [ ]:
if not skip:
    
    file_paths = [Path(file_path).relative_to(test_dataset_path).as_posix() 
                  for file_path in utils.image_dataset_from_directory(test_dataset_path, labels=None, shuffle=False, batch_size=1).file_paths]

#### Создание таблицы процесса

In [ ]:
if not skip:
    
    processing = pd.DataFrame(
        columns = ['file_path', 'image_size', 'faces_num', 'face_size', 'batch', 'iter', 'engine', 'status']
    )
    processing['file_path'] = file_paths
    processing['faces_num'] = 0
    processing['image_size'] = 0
    processing['face_size'] = 0
    processing['batch'] = processing.index // params['batch_size']
    processing['iter'] = processing['batch'] // params['engines']
    processing['engine'] = processing['batch'] % params['engines']

#### Создание таблицы результатов

In [ ]:
if not skip:
    
    test_face_extraction = pd.DataFrame(
        columns = [
            'emotion',
            'faces_num',
        ],
    )
    test_face_extraction.set_index('emotion', inplace=True)

#### Инициализация параллельных вычислений

In [ ]:
if not skip:
    # Создаем кластер
    cluster = ipp.Cluster()
    # Запускаем контроллер
    await cluster.start_controller()
    # Запускаем движки по размеру батча
    engine_set_id = await cluster.start_engines(n=params['engines'])
    # Создаем клиента
    rc = await cluster.connect_client()
    # Подключаем клиента к движкам
    rc.wait_for_engines(params['engines'])
    # Создаем наблюдателя
    view = rc.load_balanced_view()

#### Извлечение изображений лиц из тестового датасета

In [ ]:
if not skip:
    
    with trange(processing.shape[0], unit='file') as t:
        # Идем по итерациям аугментаций
        for iter_index, iter_batches in processing.groupby('iter'):
            # Создаем список результатов выполнения аугментации "движками"
            async_results = []
            # Запускаем аугментацию батча файлов на каждом "движке"
            for engine_index, engine_batch in iter_batches.groupby('engine'):
                # Получаем батч файлов
                file_paths_batch = engine_batch['file_path'].to_list()
                # Запускаем аугментацию батча файлов
                async_result = view.apply_async(extract_faces, test_dataset_path, test_faces_dataset_path, file_paths_batch, params, False)
                async_results.append(async_result)
            # Ждем окончания обработки батчей всеми "движками"
            rc.wait(async_results)
            # Вносим результат в таблицу процесса
            for engine_index, engine_batch in iter_batches.groupby('engine'):
                processing.loc[engine_batch.index, 'status'] = async_results[engine_index].status
                processing.loc[engine_batch.index, ['image_size', 'faces_num', 'face_size']] = async_results[engine_index].result()
                if async_results[engine_index].status == 'error':
                    # При появлении ошибки завершаем обработку
                    pipeline.fail_stage()
                    break
                if sum(processing.loc[engine_batch.index, 'faces_num']) < len(engine_batch):
                    # Если не удалось найти хоть одно лицо, то завершаем обработку
                    processing.loc[engine_batch.index, 'status'] = 'failed'
                    pipeline.fail_stage()
                    break

            # Обновляем счётчик Progress Bar
            t.update(iter_batches.shape[0])

            # Завершение при появлении ошибки
            if pipeline.is_stage_failed:
                break

#### Остановка параллельных вычислений

In [ ]:
if not skip:

    await cluster.stop_engines(engine_set_id)
    await cluster.stop_cluster()

#### Сохранение изображений лиц тренировочного датасета в архиве на Google Диск

In [ ]:
if not skip:
    
    if not pipeline.is_stage_failed:
        shutil.make_archive(gd_proj_path / test_faces_dataset_path.name, 'zip', test_faces_dataset_path)

#### Сохранение результатов на Google Диск

In [ ]:
if not skip:
    
    pipeline.save_stage_processing(processing)
    if not pipeline.is_stage_failed:
        faces_num = sum(processing['faces_num'] > 0)
        test_face_extraction.loc['all', 'faces_num'] = faces_num
        test_face_extraction = test_face_extraction.astype(int)
        pipeline.save_stage_result(test_face_extraction)

#### Фиксация завершения этапа

In [ ]:
if not skip and not pipeline.is_stage_failed:

    pipeline.complete_stage()

#### Загрузка результатов из Google Диск (выполняется при пропуске этапа)

In [ ]:
if skip:
    
    if pipeline.is_stage_complete:
        test_face_extraction = pipeline.load_stage_result().set_index('emotion')

#### Вывод результатов

In [ ]:
if pipeline.is_stage_complete:
    display(test_face_extraction)

,faces_num
emotion,
all,5000


### Балансировка тренировочного датасета

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params
train_faces_balanced_dataset_path = Path(params['path'])

#### Считывание датасета из архива на Google Диск

In [ ]:
if not skip:
    
    if not train_faces_dataset_path.exists():
        train_faces_dataset_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / train_faces_dataset_path.with_suffix('.zip').name,
            train_faces_dataset_path
        )

#### Копирование файлов изображений лиц в тренировочном датасете

In [ ]:
if not skip:
    
    if train_faces_balanced_dataset_path.exists():
        shutil.rmtree(train_faces_balanced_dataset_path)
    train_faces_balanced_dataset_path.mkdir()
    for emotion in EMOTIONS.keys():
        (train_faces_balanced_dataset_path / emotion).mkdir()

#### Получение списка файлов изображений лиц в тренировочном датасете

In [ ]:
if not skip:
    
    file_paths = []
    for file_path in train_faces_dataset_path.glob('**/*.*'):
        file_paths.append(Path(file_path).relative_to(train_faces_dataset_path).as_posix())

#### Создание таблицы процесса

In [ ]:
if not skip:
    
    processing = pd.DataFrame(
        columns=['file_path', 'copies_number', 'done']
    )
    processing['file_path'] = file_paths
    for emotion in EMOTIONS.keys():
        processing.loc[processing['file_path'].str.contains(emotion), 'emotion'] = emotion
    processing['copies_number'] = 0

#### Создание таблицы результатов

In [ ]:
if not skip:
    
    balancing = pd.DataFrame(
        columns=['emotion', 'initial_size', 'addition_size']
    )
    balancing['emotion'] = list(EMOTIONS.keys())
    balancing.set_index('emotion', inplace=True)
    # Получаем размеры классов
    balancing['initial_size'] = processing.groupby(['emotion']).size()
    # Рассчитываем сколько изображений нужно добавить для баланса
    balancing['addition_size'] = max(balancing['initial_size']) - balancing['initial_size']

#### Создание списка файлов и количества необходимых копий каждого файла

In [ ]:
if not skip:
    
    for emotion, emotion_processing in processing.groupby(['emotion']):
        addition_size = balancing.loc[emotion, 'addition_size']
        while addition_size > len(emotion_processing):
            processing.loc[emotion_processing.index, 'copies_number'] += 1
            addition_size -= len(emotion_processing)
        sample = emotion_processing.sample(addition_size, random_state=SEED)
        processing.loc[sample.index, 'copies_number'] += 1

#### Создание копий файлов лиц тренировочного датасета

In [ ]:
if not skip:
    
    with tqdm(processing, unit='file') as t:
        # Идем по итерациям аугментаций
        for _, (file_path, copies_number) in processing[['file_path', 'copies_number']].iterrows():
            source = train_faces_dataset_path / file_path
            for copy in range(copies_number + 1):
                dest = train_faces_balanced_dataset_path / file_path
                dest = dest.with_name(f'{copy}_' + dest.name)
                shutil.copyfile(source, dest)
            # Обновляем счётчик Progress Bar
            t.update()

#### Сохранение сбалансированного тренировочного датасета изображений лиц в архиве на Google Диск

In [ ]:
if not skip:
    
    pipeline.save_stage_processing(processing)
    shutil.make_archive(gd_proj_path / train_faces_balanced_dataset_path.name, 'zip', train_faces_balanced_dataset_path)

#### Сохранение результатов на Google Диск

In [ ]:
if not skip:
    
    pipeline.save_stage_result(balancing)

#### Фиксация завершения этапа

In [ ]:
if not skip:
    
    pipeline.complete_stage()

#### Загрузка результатов из Google Диск (выполняется при пропуске этапа)

In [ ]:
if skip:
        
    if pipeline.is_stage_complete:
        balancing = pipeline.load_stage_result().set_index('emotion')

#### Вывод результатов

In [ ]:
if pipeline.is_stage_complete:
    display(balancing)

,initial_size,addition_size
emotion,,
anger,6981,0
contempt,3070,3911
disgust,3137,3844
fear,5010,1971
happy,5926,1055
neutral,6742,239
sad,6705,276
surprise,6288,693
uncertain,5877,1104


### Отчёт о выполнении пайплайна

In [ ]:
display(pipeline.report)

,params,platform,start_time,update_time,state
stage,,,,,
train_face_extraction,"{'path': 'train_faces', 'engines': 8, 'batch_s...",local,2022-10-23 10:19:52.047153,2022-10-23 11:27:25.583180,complete
test_face_extraction,"{'path': 'test_faces', 'engines': 8, 'batch_si...",local,2022-10-23 11:27:25.743022,2022-10-23 11:36:22.142532,complete
train_balancing,"{'path': 'train_balanced_faces', 'engines': 8,...",local,2022-10-23 11:36:22.309349,2022-10-23 11:39:11.561157,complete


## Пайплайн сбора информации о базовых моделях в Keras Applications

### Создание/загрузка пайплайна из Google Диск

In [ ]:
pipeline = Pipeline(config=KERAS_BASE_MODELS_PROCESSING_PIPELINE, proj_path=gd_proj_path, is_prev_complete=pipeline.is_complete, platform=platform)

### Получение информации о размерах входных изображений и векторов признаков

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params

#### Создание таблицы результатов

In [ ]:
if not skip:
    
    base_models_sizes = pd.DataFrame(columns=['base_model_name', 'image_size', 'feature_size'])
    base_models_sizes['base_model_name'] = KERAS_BASE_MODELS.keys()
    base_models_sizes.set_index('base_model_name', inplace=True)

#### Получение информации о размерах изображений и признаков моделей

In [ ]:
if not skip:
    
    with tqdm(base_models_sizes.index, unit='model') as t:
        for base_model_name in t:
            t.set_description(f'{base_model_name}')

            # Получаем размер входного изображения
            base_model = getattr(applications, base_model_name)(include_top=True, weights=None)
            image_size = base_model.input_shape[1]
            if image_size is None:
                image_size = 299

            # Получаем размер вектора признаков
            base_model = getattr(applications, base_model_name)(include_top=False, weights=None)
            feature_size = base_model.output_shape[-1]

            # Заносим результаты в таблицу
            base_models_sizes.loc[base_model_name] = image_size, feature_size

#### Сохранение результатов на Google Диск

In [ ]:
if not skip:
    
    pipeline.save_stage_result(base_models_sizes)

#### Фиксация завершения этапа

In [ ]:
if not skip:
    
    pipeline.complete_stage()

#### Загрузка результатов из Google Диск (выполняется при пропуске этапа)

In [ ]:
if skip:
    
    if pipeline.is_stage_complete:
        base_models_sizes = pipeline.load_stage_result().set_index('base_model_name')

#### Вывод результатов

In [ ]:
if pipeline.is_stage_complete:
    display(base_models_sizes)

,image_size,feature_size
base_model_name,,
MobileNet,224,1024
MobileNetV2,224,1280
NASNetMobile,224,1056
InceptionV3,299,2048
ResNet50V2,224,2048
EfficientNetB0,224,1280
ResNet50,224,2048
EfficientNetB1,240,1280
VGG16,224,512


### Измерение времени инференса моделей

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params

#### Создание таблицы результатов

In [ ]:
if not skip:
    
    model_inference_times = pd.DataFrame(columns=['base_model_name', 'inference_time'])
    model_inference_times['base_model_name'] = KERAS_BASE_MODELS.keys()
    model_inference_times.set_index('base_model_name', inplace=True)

#### Формируем самую "тяжелую" конфигурацию полносвязной модели

In [ ]:
if not skip:
    
    model_on_top_config = [(max(MODEL_ON_TOP_DROPOUT_RATES), max(MODEL_ON_TOP_DENSE_UNITS))] * max(MODEL_ON_TOP_DENSE_NUMS)

#### Измерение времени инференса моделей

In [ ]:
if not skip:
    
    with tqdm(model_inference_times.index, unit='model') as t:
        for base_model_name in t:
            t.set_description(f'{base_model_name}')

            image_size, feature_size = base_models_sizes.loc[base_model_name, ['image_size', 'feature_size']]

            # Создаем модель со случайными весами для измерения времени инференса
            base_model = build_base_model(base_model_name, image_size=image_size, weights=None, pooling=BASE_MODEL_POOLINGS, include_preprocess_input=True)
            model_on_top = build_model_on_top(feature_size, model_on_top_config)
            model = build_model(base_model=base_model, model_on_top=model_on_top)
            model.trainable = False

            # Создаем датасет для измерения времени инференса
            dataset = tf.data.Dataset.from_tensor_slices(np.random.randint(0, 255, (params['batch_size'], image_size, image_size, 3)))
            dataset = dataset.batch(params['batch_size'])
            dataset = dataset.repeat(params['batches'])

            # Прогрев модели на одном батче
            model.predict(dataset.take(1))

            # Измеряем суммарное время инференса
            inf_time = timeit(
                lambda: model.predict(dataset), number=params['repetitions']
            )

            # Заносим результаты в таблицу
            model_inference_times.loc[base_model_name, 'colab_inference_time'] = inf_time

    # Вычисляем среднее время инференса на одном шаге
    steps_total = params['batch_size'] * params['batches'] * params['repetitions']
    model_inference_times['colab_inference_time'] /= steps_total

#### Сохранение результатов на Google Диск

In [ ]:
if not skip:
    
    pipeline.save_stage_result(model_inference_times)

#### Фиксация завершения этапа

In [ ]:
if not skip:
    
    pipeline.complete_stage()

#### Загрузка результатов из Google Диск (выполняется при пропуске этапа)

In [ ]:
if skip:
    
    if pipeline.is_stage_complete:
        model_inference_times = pipeline.load_stage_result().set_index('base_model_name')

#### Вывод результатов

In [ ]:
if pipeline.is_stage_complete:
    display(model_inference_times)

,gpu_inference_time,colab_inference_time
base_model_name,,
MobileNet,0.0226,0.019039
MobileNetV2,0.0259,0.021826
NASNetMobile,0.0270,0.049024
InceptionV3,0.0422,0.028462
ResNet50V2,0.0456,0.023039
EfficientNetB0,0.0460,0.027717
ResNet50,0.0582,0.024083
EfficientNetB1,0.0602,0.033303
VGG16,0.0695,0.017329


### Выбор базовой модели

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params

#### Отбор моделей по соотношение точности и макс. времени инференса

In [ ]:
if not skip:
    
    base_model_selection = pd.DataFrame()
    base_model_selection['base_model_name'] = KERAS_BASE_MODELS.keys()
    base_model_selection[['size', 'top1_accuracy']] = KERAS_BASE_MODELS.values()
    base_model_selection.set_index('base_model_name', inplace=True)
    base_model_selection = base_model_selection.merge(model_inference_times['inference_time'], on='base_model_name')
    base_model_selection = base_model_selection[
        (base_model_selection['size'] <= BASE_MODEL_MAX_SIZE) & (base_model_selection['inference_time'] <= MAX_INFERENCE_TIME)
    ]

#### Ранжирование моделей прошедших отбор по соотношению точности и макс. времени инференса

In [ ]:
if not skip:
    
    base_model_selection['top1_accuracy_score'] = (base_model_selection['top1_accuracy'] - min(base_model_selection['top1_accuracy'])) / (max(base_model_selection['top1_accuracy']) - min(base_model_selection['top1_accuracy']))
    base_model_selection['colab_inference_time_score'] = (max(base_model_selection['colab_inference_time']) - base_model_selection['colab_inference_time']) / (max(base_model_selection['colab_inference_time']) - min(base_model_selection['colab_inference_time']))
    base_model_selection['weighted_score'] = base_model_selection['top1_accuracy_score']*params['top1_accuracy_weight'] + base_model_selection['colab_inference_time_score']*params['inference_time_weight']
    base_model_selection['rank'] = base_model_selection['weighted_score'].rank(ascending=False).astype(int)
    base_model_selection.sort_values('rank', inplace=True)
    display(base_model_selection)

#### Выбор наиболее подходящей модели

In [ ]:
if not skip:
    
    base_model_info = base_model_sizes.loc[base_model_selection.index[0]:base_model_selection.index[0]]

#### Сохранение результатов на Google Диск

In [ ]:
if not skip:
    
    pipeline.save_stage_processing(base_model_selection)
    pipeline.save_stage_result(base_model_info)

#### Фиксация завершения этапа

In [ ]:
if not skip:

    pipeline.complete_stage()

#### Загрузка результатов из Google Диск (выполняется при пропуске этапа)

In [ ]:
if skip:
    
    if pipeline.is_stage_complete:
        base_model_info = pipeline.load_stage_result().set_index('base_model_name')

#### Вывод результатов

In [ ]:
if pipeline.is_stage_complete:
    base_model_name = base_model_info.index[0]
    image_size, feature_size = base_model_info.iloc[0]
    display(base_model_info)

,image_size,feature_size
base_model_name,,
EfficientNetB0,224,1280


### Отчёт о выполнении пайплайна

In [ ]:
display(pipeline.report)

,params,platform,start_time,update_time,state
stage,,,,,
sizes_retrieving,{'result_csv': 'base_model_sizes.csv'},colab,2022-10-24 12:26:34.168014,2022-10-24 12:29:32.516303,complete
inference_time_measuring,"{'batch_size': 1, 'batches': 1, 'repetitions':...",colab,2022-10-24 12:29:32.556841,2022-10-24 12:36:42.796269,complete
base_model_selection,"{'inference_time_weight': 0.6, 'top1_accuracy_...",colab,2022-10-24 12:36:42.837679,2022-10-24 12:36:44.442516,complete


## Пайплайн создания модели

### Создание/загрузка пайплайна из Google Диск

In [ ]:
pipeline = Pipeline(config=MODEL_BUILDING_PIPELINE, proj_path=gd_proj_path, is_prev_complete=pipeline.is_complete, platform=platform)

### Извлечение признаков из тренировочного датасета

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params
train_features_dataset_path = Path(params['path'])

#### Считывание тренировочного датасета изображений лиц из архива на Google Диск

In [ ]:
if not skip:
    
    if not train_faces_balanced_dataset_path.exists():
        train_faces_balanced_dataset_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / train_faces_balanced_dataset_path.with_suffix('.zip').name,
            train_faces_balanced_dataset_path
        )

#### Создание директории для файлов признаков

In [ ]:
if not skip:
    
    if train_features_dataset_path.exists():
        shutil.rmtree(train_features_dataset_path)
    train_features_dataset_path.mkdir()

#### Извлечение и сохранение признаков для выбранной модели

In [ ]:
if not skip:

    # Формируем датасет с изображениями оптимального для модели размера
    # Добавляем в датасет исходные изображения
    dataset = utils.image_dataset_from_directory(
        train_faces_balanced_dataset_path,
        batch_size=params['batch_size'],
        label_mode='int',
        image_size=(image_size, image_size),
        shuffle=False)

    # Количество изображений определяем по длине списка файлов
    images_cnt = len(dataset.file_paths)

    # Инициализуем предварительную подготовку следующего батча
    dataset = dataset.prefetch(buffer_size=params['buffer_size'])

    # Создаем массивы признаков и меток
    features = np.zeros(shape=(images_cnt, feature_size))
    labels = np.zeros(shape=(images_cnt, 1))

    # Создаем модель аугментации
    augment_model = build_augment_model(
        image_size=image_size,
        flip=params['flip'],
        rotation_factor=params['rotation_factor'],
        zoom_factor=params['zoom_factor'],
        contrast_factor=params['contrast_factor'],
        brightness_factor=params['brightness_factor'],
        training=True
    )

    # Создаем базовую модель, обученную на датасете ImageNet
    base_model = build_base_model(
        base_model_name, weights='imagenet', 
        image_size=image_size, 
        pooling=BASE_MODEL_POOLINGS, 
        include_preprocess_input=True,
        training=False
    )

    # Соединяем все созданные выше модели в одну
    model = build_model(augment_model=augment_model, base_model=base_model)

    # Создаем Progress Bar для отслеживания прогресса выполнения извлечения признаков
    with tqdm(dataset, desc=f'{base_model_name}', unit='batch') as t:

        # Индексы слайса вставки в массивы признаков и меток
        start_index = 0
        end_index = 0

        # Идем по батчам датасета изображений, каждый из которых состоит из батча изображений и батча меток
        for batch, (batch_images, batch_labels) in enumerate(dataset.as_numpy_iterator()):

            end_index += batch_images.shape[0]

            # Извлекаем признаки из батча изображений и вставляем в массив
            batch_features = model(batch_images)
            features[start_index: end_index] = batch_features

            # Преобразуем размерность батча меток и вставляем его в массив
            batch_labels = batch_labels.reshape(-1, 1)
            labels[start_index: end_index] = batch_labels

            start_index = end_index

            # Обновляем Progress Bar
            t.update()

    # Записываем массивы признаков и меток в файл
    feature_path = (train_features_dataset_path / base_model_name).with_suffix('.npz')
    np.savez(feature_path, features=features, labels=labels)

#### Сохранение архива файлов признаков на Google Диск

In [ ]:
if not skip:
    
    shutil.make_archive(gd_proj_path / train_features_dataset_path.name, 'zip', train_features_dataset_path)

#### Фиксация завершения этапа

In [ ]:
if not skip:
    
    pipeline.complete_stage()

### Извлечение признаков из тестового датасета

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params
test_features_dataset_path = Path(params['path'])

#### Считывание тестового датасета изображений лиц из архива на Google Диск

In [ ]:
if not skip:
    
    if not test_faces_dataset_path.exists():
        test_faces_dataset_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / test_faces_dataset_path.with_suffix('.zip').name,
            test_faces_dataset_path
        )

#### Создание директорий для файлов признаков

In [ ]:
if not skip:
    
    if test_features_dataset_path.exists():
        shutil.rmtree(test_features_dataset_path)
    test_features_dataset_path.mkdir()

#### Извлечение и сохранение признаков для выбранной модели

In [ ]:
if not skip:

    # Формируем датасет с изображениями оптимального для модели размера
    # Добавляем в датасет исходные изображения
    dataset = utils.image_dataset_from_directory(
        test_faces_dataset_path,
        batch_size=params['batch_size'],
        label_mode=None,
        image_size=(image_size, image_size),
        shuffle=False)

    # Количество изображений определяем по длине списка файлов
    images_cnt = len(dataset.file_paths)

    # Инициализуем предварительную подготовку следующего батча
    dataset = dataset.prefetch(buffer_size=params['buffer_size'])

    # Создаем массивы признаков
    features = np.zeros(shape=(images_cnt, feature_size))

    # Создаем модель аугментации
    augment_model = build_augment_model(
        image_size=image_size,
        flip=params['flip'],
        rotation_factor=params['rotation_factor'],
        zoom_factor=params['zoom_factor'],
        contrast_factor=params['contrast_factor'],
        brightness_factor=params['brightness_factor'],
        training=True
    )

    # Создаем базовую модель, обученную на датасете ImageNet
    base_model = build_base_model(
        base_model_name, weights='imagenet', 
        image_size=image_size, 
        pooling=BASE_MODEL_POOLINGS, 
        include_preprocess_input=True,
        training=False
    )

    # Соединяем все созданные выше модели в одну
    model = build_model(augment_model=augment_model, base_model=base_model)

    # Создаем Progress Bar для отслеживания прогресса выполнения извлечения признаков
    with tqdm(dataset, desc=f'{base_model_name}', unit='batch') as t:

        # Индексы слайса вставки в массивы признаков и меток
        start_index = 0
        end_index = 0

        # Идем по батчам датасета изображений, каждый из которых состоит из батча изображений и батча меток
        for batch, batch_images in enumerate(dataset.as_numpy_iterator()):

            end_index += batch_images.shape[0]

            # Извлекаем признаки из батча изображений и вставляем в массив
            batch_features = model(batch_images)
            features[start_index: end_index] = batch_features

            start_index = end_index

            # Обновляем Progress Bar
            t.update()

    # Записываем массивы признаков и меток в файл
    feature_path = (test_features_dataset_path / base_model_name).with_suffix('.npz')
    np.savez(feature_path, features=features)

#### Сохранение архива файлов признаков на Google Диск

In [ ]:
if not skip:
    
    shutil.make_archive(gd_proj_path / test_features_dataset_path.name, 'zip', test_features_dataset_path)

#### Фиксация завершения этапа

In [ ]:
if not skip:
    
    pipeline.complete_stage()

### Выбор лучшей полносвязной модели

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params
model_on_top_selection_path = Path(params['path'])
model_on_top_selection_logs_path = model_on_top_selection_path / 'logs'
model_on_top_selection_models_path = model_on_top_selection_path / 'models'

#### Считывание тренировочного датасета признаков из архива на Google Диск

In [ ]:
if not skip:
    
    if not train_features_dataset_path.exists():
        train_features_dataset_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / train_features_dataset_path.with_suffix('.zip').name,
            train_features_dataset_path
        )

#### Считывание тестового датасета признаков из архива на Google Диск

In [ ]:
if not skip:
    
    if not test_features_dataset_path.exists():
        test_features_dataset_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / test_features_dataset_path.with_suffix('.zip').name,
            test_features_dataset_path
        )

#### Считывание тестового датасета изображений лиц из архива на Google Диск

In [ ]:
if not skip:
    
    if not test_faces_dataset_path.exists():
        test_faces_dataset_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / test_faces_dataset_path.with_suffix('.zip').name,
            test_faces_dataset_path
        )

#### Создание списка возможных комбинаций гиперпараметров

In [ ]:
if not skip:
    
    # Методом перебора получаем список возможных сочетаний коэффициента регуляризации и размер полносвязного слоя
    dropout_rate_dense_units_combs = [(dropout_rate, dense_units) for dense_units in MODEL_ON_TOP_DENSE_UNITS for dropout_rate in MODEL_ON_TOP_DROPOUT_RATES]
    # Получаем список комбинаций
    model_on_top_configs = []
    for dense_num in MODEL_ON_TOP_DENSE_NUMS:
        model_on_top_configs += set(itertools.permutations(dropout_rate_dense_units_combs, dense_num)).union(set(itertools.combinations_with_replacement(dropout_rate_dense_units_combs, dense_num)))
    model_on_top_configs = sorted(model_on_top_configs)
    model_on_top_config_strs = [', '.join([str(element) for element in model_on_top_config]) for model_on_top_config in model_on_top_configs]

#### Удаление существующих логов и весов моделей

In [ ]:
if not skip:
    
    shutil.rmtree(model_on_top_selection_path, ignore_errors=True)
    model_on_top_selection_path.mkdir()
    model_on_top_selection_models_path.mkdir()
    model_on_top_selection_logs_path.mkdir()
    for model_on_top_config_str in model_on_top_config_strs:
        (model_on_top_selection_logs_path / model_on_top_config_str).mkdir()

#### Создание таблицы результатов обучения

In [ ]:
if not skip:
    
    if isinstance(EMOTIONS, (list, tuple)):
        loss = 'sparse_categorical_crossentropy'
        metric = 'sparse_categorical_accuracy'
    else:
        loss = 'mean_absolute_error'
        metric = 'mean_absolute_percentage_error'

    processing = pd.DataFrame(
        columns=[
            'model_on_top_config', 
            'best_epoch',
            'loss_at_best_epoch',
            f'{metric}_at_best_epoch',
            'best_test_score',
        ],
        dtype='float'
    )
    processing['model_on_top_config'] = model_on_top_config_strs
    processing.set_index('model_on_top_config', inplace=True)

#### Создание таблицы результатов

In [ ]:
if not skip:
    
    models_on_top = pd.DataFrame(
        columns = [
            'model_on_top_config',
            'best_test_score',
        ],
        dtype=float
    )
    models_on_top.set_index('model_on_top_config', inplace=True)

#### Создание обучающего и тестового датасета

In [ ]:
if not skip:
    
    # Создаем тренировочный датасет
    train_dataset = build_feature_dataset(
        (train_features_dataset_path / base_model_name).with_suffix('.npz').as_posix(),
        batch_size=params['batch_size'],
        shuffle=True,
        seed=SEED
    )

    # Создаем тестовый датасет
    test_dataset = build_feature_dataset(
        (test_features_dataset_path / base_model_name).with_suffix('.npz').as_posix(),
        labeled=False,
        batch_size=params['batch_size'],
        shuffle=False,
    )

    # Создаем список имен файлов изображений в тестовом датасете
    test_image_paths = [Path(image_path).name for image_path in utils.image_dataset_from_directory(test_faces_dataset_path, label_mode=None, shuffle=False).file_paths]

#### Обучение полносвязных моделей

In [ ]:
if not skip:

    # Запускаем логгирование для TensorBoard
    %tensorboard --logdir {model_on_top_selection_logs_path.as_posix()}

In [ ]:
if not skip:
    
    # Перебираем конфигурации
    with tqdm(model_on_top_configs, unit='config') as t:

        for model_on_top_config, model_on_top_config_str in zip(model_on_top_configs, model_on_top_config_strs):

            t.set_description(model_on_top_config_str)

            # Создаем и компилируем полносвязную модель
            model_on_top = build_model_on_top(
                feature_size=feature_size, 
                config=model_on_top_config,
                training=True
            )

            # Создаем оптимайзер с экспоненциальным уменьшением скорости (каждую эпоху)
            optimizer = getattr(optimizers, params['optimizer_name'])(learning_rate=params['initial_learning_rate'])

            # Компилируем модель
            model_on_top.compile(optimizer=optimizer, loss=loss, metrics=metric)

            # Снижение скорости обучения
            learning_rate_callback = LearningRateExpDecayScheduler(params['learning_rate_decay_rate']) 

            # Оставаливаем обучение, если точность на тестовых данных перестает расти
            earlystop_callback = EarlyStoppingAtMaxTestScore(
                model=model_on_top,
                emotions=EMOTIONS,
                test_dataset=test_dataset,
                test_image_paths=test_image_paths,
                patience=params['patience'],
            )

            # Отборажение графиков обучения в TensorBoard
            tensorboard_callback = callbacks.TensorBoard(
                log_dir=model_on_top_selection_logs_path / model_on_top_config_str,
                update_freq="epoch",
            )

            # Обновление информации Progress Bar
            epoch_end_callback = callbacks.LambdaCallback(
                on_epoch_end=lambda epoch, logs: t.set_description(
                    f"{model_on_top_config_str} [#{epoch+1}: " 
                    f"{loss}={logs['loss']:.4f}, "
                    f"{metric}={logs[metric]:.4f}, "
                    f"lr={logs['lr']:.4e}, "
                    f"public_score={logs['test_public_score']:.4f}, "
                    f"private_score={logs['test_private_score']:.4f}, "
                    f"score={logs['test_score']:.4f}]"
                )
            )

            # Обучаем полносвязную модель
            history = model_on_top.fit(
                train_dataset,
                epochs=params['epochs'],
                verbose=VERBOSE,
                callbacks=[learning_rate_callback, earlystop_callback, tensorboard_callback, epoch_end_callback],
            )

            model_on_top.set_weights(earlystop_callback.best_weights)
            model_on_top.save(model_on_top_selection_models_path / model_on_top_config_str / 'best_model.h5', include_optimizer=False)

            # По истории находим лучшее значение метрики точности на тестовом датасете и заносим в таблицу процесса
            processing.loc[model_on_top_config_str, 'best_epoch'] = earlystop_callback.best_epoch
            processing.loc[model_on_top_config_str, 'loss_at_best_epoch'] = history.history['loss'][earlystop_callback.best_epoch]
            processing.loc[model_on_top_config_str, f'{metric}_at_best_epoch'] = history.history[metric][earlystop_callback.best_epoch]
            processing.loc[model_on_top_config_str, 'best_test_score'] = earlystop_callback.best_test_score

            # Обновляем Progress Bar
            t.update()

#### Выбор конфигурации полносвязной модели, обеспечивающей лучшую точность на тестовых данных

In [ ]:
if not skip:
    
    processing['best_epoch'] = processing['best_epoch'].astype(int)
    model_on_top = processing.sort_values('best_test_score', ascending=False)[['best_test_score']].head(1)

#### Сохранение логов обучения и весов моделей в архиве на Google Диск

In [ ]:
if not skip:
    
    pipeline.save_stage_processing(processing)
    shutil.make_archive(gd_proj_path / model_on_top_selection_path.name, 'zip', model_on_top_selection_path)

#### Сохранение результатов обучения на Google Диск

In [ ]:
if not skip:
    
    pipeline.save_stage_result(model_on_top)

#### Фиксация завершения этапа

In [ ]:
if not skip:
    
    pipeline.complete_stage()

#### Загрузка результатов обучения из Google Диск (выполняется при пропуске этапа)

In [ ]:
if skip:
    
    if pipeline.is_stage_complete:
        model_on_top = pipeline.load_stage_result().set_index('model_on_top_config')

#### Вывод результатов

In [ ]:
if pipeline.is_stage_complete:
    model_on_top_config_str = model_on_top.index[0]
    model_on_top_config_substrs = [element.split(', ') for element in model_on_top_config_str[1:-1].split('), (')]
    model_on_top_config = [(float(dropout_rate_str), int(dense_units_str)) for dropout_rate_str, dense_units_str in model_on_top_config_substrs]
    display(model_on_top)

,best_test_score
model_on_top_config,
"(0.0, 2048)",0.3792


### Обучение полносвязной модели

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params
model_on_top_train_path = Path(params['path'])
model_on_top_train_logs_path = model_on_top_train_path / 'logs'
model_on_top_train_models_path = model_on_top_train_path / 'models'

#### Считывание изображений лиц тренировочного датасета из архива на Google Диск

In [ ]:
if not skip:
    
    if not train_faces_balanced_dataset_path.exists():
        train_faces_balanced_dataset_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / train_faces_balanced_dataset_path.with_suffix('.zip').name,
            train_faces_balanced_dataset_path
        )

#### Считывание изображений лиц тестового датасета из архива на Google Диск

In [ ]:
if not skip:
    
    if not test_faces_dataset_path.exists():
        test_faces_dataset_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / test_faces_dataset_path.with_suffix('.zip').name,
            test_faces_dataset_path
        )

#### Считывание результатов выбора полносвязной модели из архива на Google Диск

In [ ]:
if not skip:
    
    if not model_on_top_selection_path.exists():
        model_on_top_selection_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / model_on_top_selection_path.with_suffix('.zip').name,
            model_on_top_selection_path
        )

#### Удаление существующих логов и весов модели (выполняется при первой итерации)

In [ ]:
if not skip:
    
    # Удаляем папку для логов и весов моделей при первом проходе
    if pipeline.is_stage_started:
        shutil.rmtree(model_on_top_train_path, ignore_errors=True)

    # Создаем папку для логов и весов моделей
    if not model_on_top_train_path.exists():
        model_on_top_train_path.mkdir()
        model_on_top_train_logs_path.mkdir()
        model_on_top_train_models_path.mkdir()

#### Загрузка логов и весов модели с предыдущей итерации (выполняется при второй и последующей итерации)

In [ ]:
if not skip:
    
    if not pipeline.is_stage_started:
        # Считываем архив предыдущих проходов из архива на Google Диск'е
        shutil.unpack_archive(
            gd_proj_path / model_on_top_train_path.with_suffix('.zip').name,
            model_on_top_train_path
        )

#### Создание таблицы результатов обучения

In [ ]:
if not skip:
    
    model_on_top_train = pd.DataFrame(
        columns=[
            'base_model_name',
            'model_on_top_config',
            'best_epoch',
            'best_test_score',
        ],
        dtype=np.int32
    )
    model_on_top_train.loc[0, 'base_model_name'] = base_model_name
    model_on_top_train.loc[0, 'model_on_top_config'] = model_on_top_config_str
    model_on_top_train.set_index(['base_model_name', 'model_on_top_config'], inplace=True)

#### Инициализация итерации

In [ ]:
if not skip:
    
    if pipeline.is_stage_started:
        # Первая итерация
        initial_epoch = 0
        initial_learning_rate = params['initial_learning_rate']
    else:
        # Вторая или последующая итерация
        # Находим номер начальной эпохи по протоколу
        processing = pipeline.load_stage_processing().set_index('epoch')
        last_epoch = processing.index[-1]
        initial_epoch = last_epoch + 1
        initial_learning_rate = processing.loc[last_epoch, 'lr']
        best_epoch = processing['test_score'].idxmax()
        best_test_score = processing.loc[best_epoch, 'test_score']
        best_weights = models.load_model(model_on_top_train_models_path / f'best_model.h5').get_weights()
        wait = last_epoch - best_epoch

    # Вычисляем номер последней эпохи
    final_epoch = min(initial_epoch + params['epochs_per_run'], params['epochs'])

#### Создание тренировочного и тестового датасета

In [ ]:
if not skip:
    
    # Создаем тренировочный датасет
    train_dataset = utils.image_dataset_from_directory(
        train_faces_balanced_dataset_path,
        batch_size=params['batch_size'],
        label_mode='int',
        image_size=(image_size, image_size),
        shuffle=True,
        seed=SEED,
    )

    # Если обучение производится в режиме Valence-Arousal, то метки эмоций заменяем на пары значений valence, arousal
    if isinstance(EMOTIONS, dict):
        def c(images, emotion_labels):
            va_labels = [list(EMOTIONS.values())[emotion_label] for emotion_label in emotion_labels]
            return images, va_labels

        train_dataset = train_dataset.map(
            lambda images, emotion_labels: tf.py_function(
                func=c, inp=(images, emotion_labels), Tout=(tf.float32, tf.float32)
            )
        )

    # Инициализуем предварительную подготовку следующего батча
    train_dataset = train_dataset.prefetch(buffer_size=params['buffer_size'])

    # Создаем тестовый датасет
    test_dataset = utils.image_dataset_from_directory(
        test_faces_dataset_path,
        batch_size=params['batch_size'],
        label_mode=None,
        image_size=(image_size, image_size),
        shuffle=False,
    )

    # Создаем список имен файлов изображений в тестовом датасете
    test_image_paths = [Path(image_path).name for image_path in test_dataset.file_paths]

    # Инициализуем предварительную подготовку следующего батча
    test_dataset = test_dataset.prefetch(buffer_size=params['buffer_size'])

#### Создание модели для обучения полносвязной модели (выполняется при первой итерации)

In [ ]:
if not skip:
    
    if initial_epoch == 0:
        # Создаем выбранную базовую модель, обученную на датасете ImageNet
        base_model = build_base_model(
            base_model_name, 
            weights='imagenet', 
            image_size=image_size,
            pooling=BASE_MODEL_POOLINGS, 
            include_preprocess_input=True, 
            training=False
        )
        base_model.trainable=False
        # Создаем выбранную полносвязную модель
        model_on_top = build_model_on_top(
            feature_size=feature_size, 
            config=model_on_top_config,
            training=True,
        )
        # Создаем модель аугментации
        augment_model = build_augment_model(
            image_size=image_size,
            flip=params['flip'],
            rotation_factor=params['rotation_factor'],
            zoom_factor=params['zoom_factor'],
            contrast_factor=params['contrast_factor'],
            brightness_factor=params['brightness_factor'],
            training=True,
        )
        # Соединяем все созданные выше модели в одну
        model = build_model(augment_model=augment_model, base_model=base_model, model_on_top=model_on_top)

#### Загрузка последней модели для обучения полносвязной модели из предыдущей итерации (выполняется при второй и последующей итерации)

In [ ]:
if not skip:
    
    if initial_epoch > 0:
        model = models.load_model(model_on_top_train_models_path / 'last_model.h5')

#### Компиляция модели

In [ ]:
if not skip:
    
    if isinstance(EMOTIONS, (list, tuple)):
        loss = 'sparse_categorical_crossentropy'
        metric = 'sparse_categorical_accuracy'
    else:
        loss = 'mean_absolute_error'
        metric = 'mean_absolute_percentage_error'

    if initial_epoch == 0:
        # Создаем оптимайзер с экспоненциальным уменьшением скорости (каждую эпоху)
        optimizer = getattr(optimizers, params['optimizer_name'])(learning_rate=initial_learning_rate)
        # Компилируем модель
        model.compile(optimizer=optimizer, loss=loss, metrics=metric)
        model.summary()

#### Обучение полносвязной модели

In [ ]:
# Запускаем логгирование для TensorBoard
if not skip:
    
    %tensorboard --logdir {model_on_top_train_logs_path.as_posix()}

In [ ]:
if not skip:
    
    # Снижение скорости обучения
    learning_rate_callback = LearningRateExpDecayScheduler(params['learning_rate_decay_rate']) 

    # Оставаливаем обучение, если точность на тестовых данных перестает расти
    if initial_epoch == 0:
        earlystop_callback = EarlyStoppingAtMaxTestScore(
            model=model,
            emotions=EMOTIONS,
            test_dataset=test_dataset,
            test_image_paths=test_image_paths,
            patience=params['patience'], 
        )
    else:
        earlystop_callback = EarlyStoppingAtMaxTestScore(
            model=model,
            emotions=EMOTIONS,
            test_dataset=test_dataset,
            test_image_paths=test_image_paths,
            patience=params['patience'], 
            best_epoch=best_epoch,
            best_test_score=best_test_score,
            best_weights=best_weights,
            wait=wait
        )

    # Отборажение графиков обучения в TensorBoard
    tensorboard_callback = callbacks.TensorBoard(
        log_dir=model_on_top_train_logs_path,
        write_graph=True,
        update_freq="epoch",
    )

    # Обновление Progress Bar в конце эпохи
    def epoch_end(epoch, logs):
        t.set_description(
            f"#{epoch+1}: "
            f"{loss}={logs['loss']:.4f}, "
            f"{metric}={logs[metric]:.4f}, "
            f"lr={logs['lr']:.4e}, "
            f"public_score={logs['test_public_score']:.4f}, "
            f"private_score={logs['test_private_score']:.4f}, "
            f"score={logs['test_score']:.4f}"
        )
        t.update()

    epoch_end_callback = callbacks.LambdaCallback(on_epoch_end=epoch_end)

    # Обучаем модель
    with trange(params['epochs'], initial=initial_epoch, desc='', unit='epoch') as t:
        history = model.fit(
            train_dataset,
            initial_epoch=initial_epoch,
            epochs=final_epoch,
            verbose=VERBOSE,
            callbacks=[learning_rate_callback, earlystop_callback, tensorboard_callback, epoch_end_callback],
        )

        # Сохраняем последнюю модель
        model.save(model_on_top_train_models_path / 'last_model.h5')

        # Сохраняем веса лучшей модели
        model.set_weights(earlystop_callback.best_weights)
        model.save(model_on_top_train_models_path / 'best_model.h5')

#### Сохранение детальной информации о процессе обучения

In [ ]:
if not skip:
    
    if initial_epoch == 0:
        processing = pd.DataFrame(history.history, index=pd.Index(history.epoch, name='epoch'))
    else:
        processing = pd.concat([processing, pd.DataFrame(history.history, index=pd.Index(history.epoch, name='epoch'))], axis=0)
    pipeline.save_stage_processing(processing)
    shutil.make_archive(gd_proj_path / model_on_top_train_path.name, 'zip', model_on_top_train_path)

#### Сохранение результатов

In [ ]:
if not skip:
    
    if (final_epoch == params['epochs']) or model.stop_training:
        model_on_top_train.loc[(base_model_name, model_on_top_config_str), 'best_epoch'] = earlystop_callback.best_epoch
        model_on_top_train['best_epoch'] = model_on_top_train['best_epoch'].astype(int)
        model_on_top_train.loc[(base_model_name, model_on_top_config_str), 'best_test_score'] = earlystop_callback.best_test_score
        pipeline.save_stage_result(model_on_top_train)

#### Фиксация завершения этапа итерации/этапа

In [ ]:
if not skip:
    
    if (final_epoch == params['epochs']) or model.stop_training:
        # Обучение закончено
        pipeline.complete_stage()
    else:
        # Завершена очередная итерация
        pipeline.complete_stage_run()

#### Загрузка результатов из Google Диск (выполняется при пропуске этапа)

In [ ]:
if skip:
    
    if pipeline.is_stage_complete:
        model_on_top_train = pipeline.load_stage_result().set_index(['base_model_name', 'model_on_top_config'])
        # model_on_top_train['best_epoch'] = model_fine_tuning['best_epoch'].astype(int)

#### Вывод результатов

In [ ]:
if pipeline.is_stage_complete:
    display(model_on_top_train)

,,best_epoch,best_test_score
base_model_name,model_on_top_config,,
EfficientNetB0,"(0.0, 2048)",1,0.3338


### Тонкая настройка модели

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params
model_fine_tuning_path = Path(params['path'])
model_fine_tuning_logs_path = model_fine_tuning_path / 'logs'
model_fine_tuning_models_path = model_fine_tuning_path / 'models'

#### Считывание изображений лиц тренировочного датасета из архива на Google Диск

In [ ]:
if not skip:
    
    if not train_faces_balanced_dataset_path.exists():
        train_faces_balanced_dataset_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / train_faces_balanced_dataset_path.with_suffix('.zip').name,
            train_faces_balanced_dataset_path
        )

#### Считывание изображений лиц тестового датасета из архива на Google Диск

In [ ]:
if not skip:
    
    if not test_faces_dataset_path.exists():
        test_faces_dataset_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / test_faces_dataset_path.with_suffix('.zip').name,
            test_faces_dataset_path
        )

#### Считывание результатов обучения полносвязной модели из архива на Google Диск

In [ ]:
if not skip:
    
    if pipeline.is_stage_started and not model_on_top_train_path.exists():
        model_on_top_train_path.mkdir()
        shutil.unpack_archive(
            gd_proj_path / model_on_top_train_path.with_suffix('.zip').name,
            model_on_top_train_path
        )

#### Удаление существующих логов и весов модели (выполняется при первой итерации)

In [ ]:
if not skip:
    
    # Удаляем папку для логов и весов моделей при первом проходе
    if pipeline.is_stage_started:
        shutil.rmtree(model_fine_tuning_path, ignore_errors=True)

    # Создаем папку для логов и весов моделей
    if not model_fine_tuning_path.exists():
        model_fine_tuning_path.mkdir()
        model_fine_tuning_logs_path.mkdir()
        model_fine_tuning_models_path.mkdir()

#### Загрузка логов и весов модели с предыдущей итерации (выполняется при второй и последующей итерации)

In [ ]:
if not skip:
    
    if not pipeline.is_stage_started:
        # Считываем архив предыдущих проходов из архива на Google Диск'е
        shutil.unpack_archive(
            gd_proj_path / model_fine_tuning_path.with_suffix('.zip').name,
            model_fine_tuning_path
        )

#### Создание таблицы результатов обучения

In [ ]:
if not skip:
    
    model_fine_tuning = pd.DataFrame(
        columns=[
            'base_model_name',
            'model_on_top_config',
            'best_epoch',
            'best_test_score',
        ],
        dtype='int'
    )
    model_fine_tuning.loc[0, 'base_model_name'] = base_model_name
    model_fine_tuning.loc[0, 'model_on_top_config'] = model_on_top_config_str
    model_fine_tuning.set_index(['base_model_name', 'model_on_top_config'], inplace=True)

#### Инициализация итерации

In [ ]:
if not skip:
    
    if pipeline.is_stage_started:
        # Первая итерация
        initial_epoch = 0
        initial_learning_rate = params['initial_learning_rate']
    else:
        # Вторая или последующая итерация
        # Находим номер начальной эпохи по протоколу
        processing = pipeline.load_stage_processing().set_index('epoch')
        last_epoch = processing.index[-1]
        initial_epoch = last_epoch + 1
        initial_learning_rate = processing.loc[last_epoch, 'lr']
        best_epoch = processing['test_score'].idxmax()
        best_test_score = processing.loc[best_epoch, 'test_score']
        best_weights = models.load_model(model_fine_tuning_models_path / f'best_model.h5').get_weights()
        wait = last_epoch - best_epoch

    # Вычисляем номер последней эпохи
    final_epoch = min(initial_epoch + params['epochs_per_run'], params['epochs'])

#### Создание тренировочного и тестового датасета

In [ ]:
if not skip:
    
    # Создаем тренировочный датасет
    train_dataset = utils.image_dataset_from_directory(
        train_faces_balanced_dataset_path,
        batch_size=params['batch_size'],
        label_mode='int',
        image_size=(image_size, image_size),
        shuffle=True,
        seed=SEED,
    )

    # Если обучение производится в режиме Valence-Arousal, то метки эмоций заменяем на пары значений valence, arousal
    if isinstance(EMOTIONS, dict):
        def c(images, emotion_labels):
            va_labels = [list(EMOTIONS.values())[emotion_label] for emotion_label in emotion_labels]
            return images, va_labels

        train_dataset = train_dataset.map(
            lambda images, emotion_labels: tf.py_function(
                func=c, inp=(images, emotion_labels), Tout=(tf.float32, tf.float32)
            )
        )

    # Инициализуем предварительную подготовку следующего батча
    train_dataset = train_dataset.prefetch(buffer_size=params['buffer_size'])

    # Создаем тестовый датасет
    test_dataset = utils.image_dataset_from_directory(
        test_faces_dataset_path,
        batch_size=params['batch_size'],
        label_mode=None,
        image_size=(image_size, image_size),
        shuffle=False,
    )

    # Создаем список имен файлов изображений в тестовом датасете
    test_image_paths = [Path(image_path).name for image_path in test_dataset.file_paths]

    # Инициализуем предварительную подготовку следующего батча
    test_dataset = test_dataset.prefetch(buffer_size=params['buffer_size'])

#### Загрузка модели для тонкой настройки

In [ ]:
if not skip:
    
    if initial_epoch == 0:
        # При первой итерации загружаем лучшую модель для обучения полносвязной модели
        model = models.load_model(model_on_top_train_models_path / 'best_model.h5')
        model.trainable = True
    else:
        # При второй и последующей итерации загружаем последнюю модель из прошлой итерации
        model = models.load_model(model_fine_tuning_models_path / 'last_model.h5')

#### Компиляция для тонкой настройки

In [ ]:
if not skip:
    
    if isinstance(EMOTIONS, (list, tuple)):
        loss = 'sparse_categorical_crossentropy'
        metric = 'sparse_categorical_accuracy'
    else:
        loss = 'mean_absolute_error'
        metric = 'mean_absolute_percentage_error'

    if initial_epoch == 0:
        # Создаем оптимайзер с экспоненциальным уменьшением скорости (каждую эпоху)
        optimizer = getattr(optimizers, params['optimizer_name'])(learning_rate=initial_learning_rate)
        # Компилируем модель
        model.compile(optimizer=optimizer, loss=loss, metrics=metric)
        model.summary()

#### Тонкая настройка модели

In [ ]:
if not skip:

    # Запускаем логгирование для TensorBoard
    %tensorboard --logdir {model_fine_tuning_logs_path.as_posix()}

In [ ]:
if not skip:
    
    # Снижение скорости обучения
    learning_rate_callback = LearningRateExpDecayScheduler(params['learning_rate_decay_rate']) 

    # Оставаливаем обучение, если точность на тестовых данных перестает расти
    if initial_epoch == 0:
        earlystop_callback = EarlyStoppingAtMaxTestScore(
            model=model,
            emotions=EMOTIONS,
            test_dataset=test_dataset,
            test_image_paths=test_image_paths,
            patience=params['patience'], 
        )
    else:
        earlystop_callback = EarlyStoppingAtMaxTestScore(
            model=model,
            emotions=EMOTIONS,
            test_dataset=test_dataset,
            test_image_paths=test_image_paths,
            patience=params['patience'], 
            best_epoch=best_epoch,
            best_test_score=best_test_score,
            best_weights=best_weights,
            wait=wait,
        )

    # Отборажение графиков обучения в TensorBoard
    tensorboard_callback = callbacks.TensorBoard(
        log_dir=model_fine_tuning_logs_path,
        write_graph=True,
        update_freq="epoch",
    )

    # Обновление Progress Bar в конце эпохи
    def epoch_end(epoch, logs):
        t.set_description(
            f"#{epoch+1}: "
            f"{loss}={logs['loss']:.4f}, "
            f"{metric}={logs[metric]:.4f}, "
            f"lr={logs['lr']:.4e}, "
            f"public_score={logs['test_public_score']:.4f}, "
            f"private_score={logs['test_private_score']:.4f}, "
            f"score={logs['test_score']:.4f}"
        )
        t.update()

    epoch_end_callback = callbacks.LambdaCallback(on_epoch_end=epoch_end)

    # Обучаем модель
    with tqdm(range(params['epochs']), initial=initial_epoch, desc='', unit='epoch') as t:
        history = model.fit(
            train_dataset,
            initial_epoch=initial_epoch,
            epochs=final_epoch,
            verbose=VERBOSE,
            callbacks=[learning_rate_callback, earlystop_callback, tensorboard_callback, epoch_end_callback],
        )

        # Сохраняем последнюю модель
        model.save(model_fine_tuning_models_path / 'last_model.h5')

        # Сохраняем веса лучшей модели
        model.set_weights(earlystop_callback.best_weights)
        model.save(model_fine_tuning_models_path / 'best_model.h5')

#### Сохранение детальной информации о процессе обучения

In [ ]:
if not skip:
    
    if initial_epoch == 0:
        processing = pd.DataFrame(history.history, index=pd.Index(history.epoch, name='epoch'))
    else:
        processing = pd.concat([processing, pd.DataFrame(history.history, index=pd.Index(history.epoch, name='epoch'))], axis=0)
    pipeline.save_stage_processing(processing)
    shutil.make_archive(gd_proj_path / model_fine_tuning_path.name, 'zip', model_fine_tuning_path)

#### Сохранение результатов

In [ ]:
if not skip:
    
    if (final_epoch == params['epochs']) or model.stop_training:
        model_fine_tuning.loc[(base_model_name, model_on_top_config_str), 'best_epoch'] = earlystop_callback.best_epoch
        model_fine_tuning['best_epoch'] = model_fine_tuning['best_epoch'].astype(int)
        model_fine_tuning.loc[(base_model_name, model_on_top_config_str), 'best_test_score'] = earlystop_callback.best_test_score
        pipeline.save_stage_result(model_fine_tuning)
        # Сохраняем финальную модель
        final_model = models.Sequential(model.layers[1:])
        final_model.trainable = False
        final_model.save(gd_proj_path / 'final_model.h5', include_optimizer=False)

#### Фиксация завершения этапа итерации/этапа

In [ ]:
if not skip:
    
    if (final_epoch == params['epochs']) or model.stop_training:
        # Обучение закончено
        pipeline.complete_stage()
    else:
        # Завершена очередная итерация
        pipeline.complete_stage_run()

#### Загрузка результатов из Google Диск (выполняется при пропуске этапа)

In [ ]:
if skip:
    
    if pipeline.is_stage_complete:
        model_fine_tuning = pipeline.load_stage_result().set_index(['base_model_name', 'model_on_top_config'])

#### Вывод результатов

In [ ]:
if pipeline.is_stage_complete:
    display(model_fine_tuning)

,,best_epoch,best_test_score
base_model_name,model_on_top_config,,
EfficientNetB0,"(0.0, 2048)",12,0.511395


### Тестирование работы модели

In [ ]:
pipeline.next_stage()
skip = pipeline.is_stage_skipped
params = pipeline.stage_params
model_deploy_test_path = Path(params['path'])

#### Удаление существующих файлов изображений

In [ ]:
if not skip:
    
    shutil.rmtree(model_deploy_test_path, ignore_errors=True)
    model_deploy_test_path.mkdir()

#### Создание таблицы процесса

In [ ]:
if not skip:
    
    if isinstance(EMOTIONS, (list, tuple)):
        processing = pd.DataFrame(
            columns=[
                'emotion',
                'image',
                'probability',
            ]
        )
    else:
        processing = pd.DataFrame(
            columns=[
                'emotion',
                'image',
                'probability',
                'valence',
                'arousal',
            ]
        )
    processing['emotion'] = list(EMOTIONS)
    processing['probability'] = 0.
    processing.set_index('emotion', inplace=True)

#### Создание таблицы результатов

In [ ]:
if not skip:
    
    if isinstance(EMOTIONS, (list, tuple)):
        model_deploy_test = pd.DataFrame(
            columns=[
                'emotion',
                'image_path',
                'probability',
            ]
        )
    else:
        model_deploy_test = pd.DataFrame(
            columns=[
                'emotion',
                'image_path',
                'error',
                'valence',
                'arousal',
            ]
        )
    model_deploy_test.set_index('emotion', inplace=True)

#### Создаем дектектор лица

In [ ]:
if not skip:
    
    face_detector = MTCNN(scale_factor=params['scale_factor'], min_face_size=params['min_face_size'])

#### Создаем модель предсказания эмоции

In [ ]:
if not skip:
    
    model = FaceEmotionRecognitionNet(
        gd_proj_path / 'final_model.h5', 
        emotions=EMOTIONS
    )

#### Инициализируем работу с камерой

In [ ]:
if not skip:
    
    cam = cv.VideoCapture(0)
    display_handle = display(display_id=True)

#### Сохраняем лучшие предсказания эмоций моделью 

In [ ]:
if not skip:
    
    while(True):
        # Делаем BRG-скриншот изображения камеры
        ret, frame = cam.read()
        if not ret:
            break
        # Преобразуем скриншот к формату RGB
        frame_rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
        # Детектируем изображение лица на скриншоте
        faces = face_detector.detect_faces(np.asarray(frame_rgb))
        if len(faces) == 0:
            continue
        x, y, w, h = faces[0]['box']
        # Вырезаем область лица из скриншота и предсказываем эмоцию
        face_boundingbox_rgb = frame_rgb[y:y + h, x:x + w]
        if isinstance(EMOTIONS, (list, tuple)):
            # Получаем предсказание эмоции и его вероятность
            emotion, probability = model.predict(face_boundingbox_rgb)
            # Запоминаем изображение лица, если эмоция была определена с самой высокой вероятностью
            if probability > processing.loc[emotion, 'probability']:
                processing.loc[emotion, 'image'] = Image.fromarray(face_boundingbox_rgb)
                processing.loc[emotion, 'probability'] = probability
        else:
            # Получаем предсказание эмоции, valence-arousal и его ошибку
            emotion, error, valence, arousal = model.predict(face_boundingbox_rgb)
            # Запоминаем изображение лица, если ошибка valence-arousal была определена с самой низкой ошибкой
            if error < processing.loc[emotion, 'error']:
                processing.loc[emotion, 'valence'] = error
                processing.loc[emotion, 'valence'] = valence
                processing.loc[emotion, 'arousal'] = arousal
        # Выделяем на скриншоте область лица с помощью зеленого прямоугольника
        frame_with_boundingbox = deepcopy(frame)
        frame_with_boundingbox = cv.rectangle(
            frame_with_boundingbox, 
            (x, y), (x + w, y + h), 
            (0,255,0), 1)
        # Над прямоугольником выводим название опредленной эмоции и её вероятность
        frame_with_boundingbox_and_title = deepcopy(frame_with_boundingbox)
        title = f'{emotion} ' + (f'({probability:.2f})') if isinstance(EMOTIONS, (list, tuple)) else f'(v: {valence:.2f}, a: {arousal:.2f})'
        frame_with_boundingbox_and_title = cv.putText(
            frame_with_boundingbox_and_title, 
            title, 
            (x, y - 10),
            cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 1)
        # Для справки слева зеленым цветом выводим список распознанных эмоций, а справа красным цветом выводим список нераспознанных эмоций
        frame_with_boundingbox_and_title_and_status = deepcopy(frame_with_boundingbox_and_title)
        recognized_emotion_cnt = 0
        unrecognized_emotion_cnt = 0
        for emotion, predict in processing.iterrows():
            if isinstance(EMOTIONS, (list, tuple)):
                probability = predict['probability']
                title = f'{emotion} ({probability:.2f})'
                recognized = probability >= params['min_probability']
            else:
                error, valence, arousal = predict[['error', 'valence', 'arousal']]
                title = f'{emotion} (v: {valence:.2f}, a: {arousal:.2f})'
                recognized = error <= params['max_error']
            if recognized:
                recognized_emotion_cnt += 1
                frame_with_boundingbox_and_title_and_status = cv.putText(
                    frame_with_boundingbox_and_title_and_status, 
                    title, 
                    (10, 30*recognized_emotion_cnt+15), 
                    cv.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 0)
            else:
                unrecognized_emotion_cnt += 1
                frame_with_boundingbox_and_title_and_status = cv.putText(
                    frame_with_boundingbox_and_title_and_status, 
                    title,
                    (frame_with_boundingbox_and_title_and_status.shape[1] - 140, 30*unrecognized_emotion_cnt), 
                    cv.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 0)
        # Даем возможность закрыть окно, если все эмоции были распознаны с приемлемой вероятностью
        done = recognized_emotion_cnt == len(EMOTIONS)
        # Выводим изображение
        _, frame_with_boundingbox_and_title_and_status = cv.imencode('.jpeg', frame_with_boundingbox_and_title_and_status)
        display_handle.update(DisplayImage(data=frame_with_boundingbox_and_title_and_status.tobytes()))
        # Заканчиваем, если все эмоции были распознаны
        if recognized_emotion_cnt == len(EMOTIONS):
            break
        
    # # Закрываем окно
    cam.release()
    display_handle.update()

#### Сохранение результатов

In [ ]:
if not skip:
    
    if isinstance(EMOTIONS, (list, tuple)):
        for emotion, (image, probability) in processing.iterrows():
            image_path = model_deploy_test_path / f'{emotion}.jpg'
            model_deploy_test.loc[emotion, ['image_path', 'probability']] = image_path.as_posix(), probability
            image.save(image_path)
    else:
        for emotion, (image, probability, valence, arousal) in processing.iterrows():
            image_path = model_deploy_test_path / f'{emotion}.jpg'
            model_deploy_test.loc[emotion, ['image_path', 'error', 'valence', 'arousal']] = image_path.as_posix(), error, valence, arousal
            image.save(image_path)

#### Сохранение результатов тестирования полносвязных моделей на Google Диск

In [ ]:
if not skip:
    
    shutil.make_archive(gd_proj_path / model_deploy_test_path.name, 'zip', model_deploy_test_path)
    pipeline.save_stage_processing(processing)
    pipeline.save_stage_result(model_deploy_test)

#### Фиксация завершения этапа

In [ ]:
if not skip:
    
    pipeline.complete_stage()

#### Загрузка оптимальной конфигурации из Google Диск (выполняется при пропуске этапа)

In [ ]:
if skip:
    
    if pipeline.is_stage_complete:
        model_deploy_test = pipeline.load_stage_result().set_index('emotion')

#### Вывод результатов

In [ ]:
if pipeline.is_stage_complete:
    
    display(model_deploy_test)
    ncols = 3
    nrows = len(EMOTIONS) // ncols + (len(EMOTIONS) % ncols > 0)
    plt.figure(figsize=(ncols*5, nrows*5))
    if isinstance(EMOTIONS, (list, tuple)):
        for index, (emotion, (image_path, probability)) in enumerate(model_deploy_test.iterrows()):
            image = cv.cvtColor(cv.imread(image_path), cv.COLOR_BGR2RGB)
            plt.subplot(ncols, nrows, index+1)
            plt.imshow(image)
            plt.title(f'{emotion} ({probability:.2f})')
    else:
        for index, (emotion, (image_path, probability, valence, arousal)) in enumerate(model_deploy_test.iterrows()):
            image = cv.cvtColor(cv.imread(image_path), cv.COLOR_BGR2RGB)
            plt.subplot(ncols, nrows, index+1)
            plt.imshow(image)
            plt.title(f'{emotion} (v: {valence:.2f}, a: {arousal:.2f}')

### Отчёт о выполнении пайплайна

In [ ]:
display(pipeline.report)

,params,platform,start_time,update_time,state
stage,,,,,
train_feature_extraction,"{'path': 'train_features', 'flip': 'horizontal...",colab,2022-10-24 12:36:44.944100,2022-10-24 13:09:32.971281,complete
test_feature_extraction,"{'path': 'test_features', 'flip': 'horizontal'...",colab,2022-10-24 13:09:32.993533,2022-10-24 13:12:14.399032,complete
model_on_top_selection,"{'path': 'model_on_top_selection', 'batch_size...",colab,2022-10-24 13:12:14.418113,2022-10-24 15:09:41.568656,complete
model_on_top_training,"{'path': 'model_on_top_training', 'flip': 'hor...",colab,2022-10-25 15:40:53.108042,2022-10-25 16:05:56.938329,complete
model_fine_tuning,"{'path': 'model_fine_tuning', 'batch_size': 32...",colab,2022-10-26 05:38:36.979050,2022-10-26 13:00:17.481069,complete
model_deploy_test,"{'path': 'model_deploy_test', 'scale_factor': ...",colab,NaN,2022-10-26 17:22:49.927267,skipped (platform)
